In [ ]:
from pathlib import Path
import importlib.util

import matplotlib as mpl
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from matplotlib.colors import LogNorm
from tokenizers import Tokenizer
from transformers import AutoConfig, AutoModelForCausalLM, PreTrainedTokenizerFast

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
    "font.size": 8,
    "axes.linewidth": 1,
    "xtick.major.width": 1,
    "ytick.major.width": 1,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "figure.dpi": 300,
    "axes.unicode_minus": False,
})


In [ ]:
def sequence_likelihood_no_bos_short(model, token_ids, device=None, max_ctx=None):
    model.eval()
    if device is None:
        device = next(model.parameters()).device

    token_ids = list(map(int, token_ids))
    N = len(token_ids)
    if N == 0: raise ValueError("token_ids is empty")

    if max_ctx is None:
        max_ctx = getattr(model.config, "n_positions", None) or getattr(model.config, "max_position_embeddings", None) or 4096
    if N > max_ctx: raise ValueError(f"Sequence length {N} exceeds model context {max_ctx}")

    input_ids = torch.tensor([token_ids], dtype=torch.long, device=device)
    with torch.no_grad():
        outputs = model(input_ids=input_ids)
        logits = outputs.logits.float()
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = input_ids[:, 1:].contiguous()
        log_probs = F.log_softmax(shift_logits, dim=-1)
        gathered = log_probs.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1).squeeze(0)
        
        per_token_logprobs = [0.0] * N
        for i, lp in enumerate(gathered.tolist(), start=1):
            per_token_logprobs[i] = float(lp)

    total_logprob = float(sum(per_token_logprobs))
    n_predicted = max(0, N - 1)
    
    if n_predicted == 0:
        avg_neg_loglik, perplexity = 0.0, 1.0
    else:
        avg_neg_loglik = - total_logprob / n_predicted
        perplexity = float(torch.exp(torch.tensor(avg_neg_loglik)).item())

    return {
        "total_logprob": total_logprob,
        "perplexity": float(perplexity)
    }


In [ ]:
PROJECT_ROOT = Path("..")
MODEL_DIR = PROJECT_ROOT / "model" / "GenNA"
TOKENIZER_JSON_PATH = PROJECT_ROOT / "configs" / "tokenizer.json"

MAX_LENGTH = 4096

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16 if device.type == "cuda" else torch.float32

FLASH_ATTN_AVAILABLE = (
    device.type == "cuda" and importlib.util.find_spec("flash_attn") is not None
)

if not MODEL_DIR.exists():
    raise FileNotFoundError(f"Required path not found: {MODEL_DIR}")

if not TOKENIZER_JSON_PATH.exists():
    raise FileNotFoundError(f"Required path not found: {TOKENIZER_JSON_PATH}")


def build_tokenizer(tokenizer_path: Path) -> PreTrainedTokenizerFast:
    tokenizer_obj = Tokenizer.from_file(str(tokenizer_path))
    tokenizer = PreTrainedTokenizerFast(
        tokenizer_object=tokenizer_obj,
        unk_token="<unk>",
        pad_token="<pad>",
        eos_token="<eos>",
        model_max_length=MAX_LENGTH,
    )
    return tokenizer


def load_model(model_dir: Path, device: torch.device, dtype: torch.dtype):
    config = AutoConfig.from_pretrained(model_dir)
    model_kwargs = {
        "config": config,
        "torch_dtype": dtype,
        "low_cpu_mem_usage": True,
    }
    if FLASH_ATTN_AVAILABLE:
        model_kwargs["attn_implementation"] = "flash_attention_2"
    model = AutoModelForCausalLM.from_pretrained(model_dir, **model_kwargs).to(device)
    model.eval()
    return model


tokenizer = build_tokenizer(TOKENIZER_JSON_PATH)
model = load_model(MODEL_DIR, device=device, dtype=dtype)

print(f"Model loaded on {device}")


In [ ]:
seq1 = "RNA, Malurus melanocephalus, {}<seq><gene>GTTTCCTTCCTCCRTGTCACACCAGTCACTTCCTGCCAGTCCCACGCTCCTCTCCAGCTTTCCTGCCATTCCCGGCTGTGCTGTCAGCGCTCCCGGGTCTCTGAGCACTCGGGACGGTGTCCCCGGCCCGGGGCACAACAGGAGGCAGGGGACGGACGGGACGCCGGCACAGAGCGCCCAGCCGGGCCCGCGTTCTCGGGCGCCCCGTCACGACCGGGCACTGTCCCGGCAGCACAGCGGCACCGGCGGC<CDS>ATGGAGCCCACCATGGCAGAACTCATCCGCACCTCCACCGACGCCTACCTGTCTGGGAGCAGCATGGCGCGGCGCCCGCGGAGCAGCCGGGCCTGGCACTTTGTCCTGAGCGGGGTCCGGCGTGAGGCCGACAGCCGGGCTGGCACACGGGGCTGGGGCTACGACTCCGACGGGCAGCAGCACAGCGACTCTGATTCAGAGCCAGAGTTCTCTTCCCTGTCTCCCTCCATCCCAAGCGCCATCCCTGTGACTGGAGAGTCCTACTGCAAYTGTGAGAACCAGAGTGAAGCTCCCTACTGCTCCAGCCTGCACGCCCTGCACCGCGTCCGTGACTGCCGCTGCGGCGAGGAGGATGAGTATTTTGACTGGGTGTGGGACGACCTGAACAAGTCGACGGCCACACTGCTGACATGTGACAATCGCAAGGTGAACTTCCACATGGAATACAGCTGTGGCACGGCTGCCATCCGCGGCAACAAGGAGCTGGCAGACGGGCAGCACTTCTGGGAGATCAAGATGACCTCCCCAGTGTATGGCACCGACATGATGGTGGGAATTGGGACATCAGATGTGAACCTGGACAAGTACCGGCACACCTTCTGCAGCCTGCTGGGCAAGGATGAGGACAGCTGGGGACTCTCCTACACAGGACTGCTGCATCACAAAGGTGACAAAACAAACTTCTCCTCGAGGTTTGGCCAAGGCTCCATTATYGGGGTGCATTTGGACACGTGGCACGGGACACTCACATTCTTCAAGAACCGCAAGTGCATAGGGGTGGCAGCCACCAAGCTGCAGAACAAGAAGTTTTACCCCATGGTGTGCTCAACAGCAGCCAAGAGCAGCATGAAGGTGATCCGCTCGTGTGCCAGCTGCACGTCATTGCAGTACCTGTGCTGCTTCCGCCTGCGCCAGCTCCTTCCCAACTATGTGGACACGCTGGAGGTGCTGCCCCTGCCCCCTGGACTCAAGCAGGTGCTGCACAACAAACTGGGCTGGGTCTTGAGCATGAACTATAGCACGTCAAAGCCTTCCTCCTCCTCGTCTTCGGGCAGCGACTCGGACAGCTCGTGCAGCTCGGATGCAGAGGCCTGCCAAAGGAAGAGGTGCAGGAGGACATAA</CDS>TGGCTCTGCACGCCGGGTGAGTGGAGGTTTATTGTGTTGTCTGCAAGGGCCAACCTCTTCCTTCATGGGGACATCCATTTGTATTCRGTGGCAAAGATTTCTCCAGTGCTGTGGAAATCTTTGGCCTCGACATCCATCACAGGCATTTGGAAGAGTTGTTGGCTCCAAGAGGTGCCAGTTCTGACACAAGAAAATTGTGGTGAAGTATCTGAGTATCTTCACCTCCTGACTTGCTATCAAAGCCAGCACCTCTTCCTGAAGGATCTCTCTTCTCCCTGTTGGCACAGACAACRTTTAAGAAATTGGACTTGATGACAGGAGTTGGACTTGACAATCTTCATGGCTCCCTTCCAACTCAGGATATTCTGTGATGATTGAGGTGGCCTCCTGGTTGGAGTGTACTGGATTCCAGTCAAATAACGGACCTGACTCATCAAGACACGATCCCAGGCTAGTCCTCTCCCTCACTGTGTGTGTAGGTTTGGTGTCTGCCATGACCTCGTCCCGCTGCTGTGGGGCACAACGGAGCAGCAGCTGTAGGGAAAGGAGCCCCACAGCCCCACTGCTGGGAGCAGGTACAAGAGCTGAGCTGTGGCTGGTCTTCCTCGRGCCCCTGGGGACTTYTAATTGCTCTGCATGTTGGTTCCTGCTGCTTCTCCTCTGTCCAAAACTGTGTTAAACTGTAATAATAAAAGTAGAAGCTGTCTGTTGTTCCTCTTGGGGCCCCTGAAGGCAGGAGGCTGCAGGTGCTTGTCAAAAATAGTCTGGCCAGCAGGGTGAAGGGGAGCTTTTGGGTGCAGCAGTTGTGTGCCAGAACTGGAGGAGGTGAAGACAGCAGAGATGCCCTCATCTGCAGG</gene></seq><eos>"
seq2= "RNA, Cariama cristata, {}<seq><gene><CDS>ATGGACGAGCCTTGCGAGCGCGGTGCCTGCTCGCCAGCCGCCGGCATGAAGGAAGGAGCGGACTGGGAGTCCTGGGACTTGCCGCTGGGTTTCCTGGACGGCGCCCAGATGGGCAACAGGAGCAACATGTCCTTTCTGCAGCTCTTCAAGAACATCAACCTGGAGCGAGCCGACGGGATGCAGGGGGACGGCTCCGACGTGATGCGGATTGTGATCTCGCTGGTGTACTCCGTGGTGTGCGCCCTGGGGCTGGTGGGCAACCTGCTGGTGCTCTACCTGATGAAAAGCAAACAAGGCTGGAGAAAGTCCTCCATCAACCTCTTTGTGACCAGCCTGGCAGTGACTGACTTCCAGTTTGTGCTGACCTTGCCATTCTGGGCAGTGGAGAATGCACTGGACTTCAACTGGCTCTTTGGCAAGGCAATGTGTAAGATTGTCTCCTATGTGACAGCCATGAATATGTATGCCAGTGTCTTCTTCCTCACTGCCATGAGTGTGGCTCGATACCGCTCTGTGGCTTCAGCCTTGAAGAATCAGCTGCGAGGTGACCCACTGGGAGGCTGCTGCTCTGCCAAGTGGCTTTGTGCACTCATCTGGCTGTCGGCTATCCTGGCTTCCCTGCCCCATGCCATTTTTTCCACCACTGCCACCGTCTTTGATGATGTACTCTGCCTCGTCAAGTTCCCCGAGGGCCGAGGCAGCAATGCCCAGTTCTGGCTGGGTCTGTACCACATCCAGAAGGTGCTGCTGGGCTTCGTGGTGCCACTAACCATCATCAGCCTCTGCTACTTGCTCCTGGTGCGCTTCATCAGCGACAAGCATGTCGGTAGCAGCTGCAGTGGCCCCAGTACCAAGCGCCGCTCCAAGGTGACTAAGTCAGTGTCCATCGTGGTGCTGTCTTTCTTCTTGTGCTGGCTGCCTAACCAAGCCCTCACAACGTGGGGCATCCTCATCAAACTCAACGTTGTGCACTTCAGTACCGAGTACTTCCTCTCCCAAGTGTACCTCTTCCCCATCAGCGTGTGCCTGGCACACTCCAACAGCTGCCTCAATCCCATCCTTTACTGCCTCATGCGCAGGGAGTTTCGCAAGGCACTGAAGAGCCTCCTCTGGAGGATCACCTCACCTTCCCTCACCACCATGCGCCCGTTCACCGACACCACCAAGCCTGAGCAGGAGGAGCAGGCCCTGCACGCCTTGCTGCCTGCCCACCCTGTTGCTGCTTCTTCCCCTGCAGCTACCGTCCAGCCGGAGGTGGCCTACTACCCCCCTGGGGTGGTGATGTACAGCAGCCGCTATGACCTGCTGCCTGCTAGCTCCATGGAGCAGCGCTGCTGA</CDS></gene></seq><eos>"
seq3 = "RNA, Orycteropus afer afer, {}<seq><gene>GGTGAAGGCTTCGTAGAGTTCAGGAATGAGGCTGACTATAAGGCTGCTCTATGTCGCCATAAACAATACATGGGCAATCGTTTTATTCAAGTTCACCCAATTACTAAGAAAGGTATGCTAGAAAAGATAGATATGATTAGAAAAAGACTACAGAACTTCAGCTATGACCAGAGGGAAATGATGTTAAACCCGGAGGGGGATGTCAGCTCTGCCAAAGTCTGTGCCCACATAACAAATATTCCATTCAGCATTACCAAGGTGGATGTTCTTCAGTTCCTAGAGGGAATCCCAGTGGATGAAAATGCTGTACATGTTCTTGTTGATAACAATGGGCAAGGTCTAGGACAGGCATTGGTTCAATTTAAGA<CDS>ATGAAGATGATGCACATGGCCCGCTGCGTGACCTTGGTTCAGCTGTCCATTTCCTGTGACCACCTCATAGACAAAGACATCGGCTCCAAGTCAGACCCACTCTGCGTTCTATTACAGGATGTGGGAGGGGGCAACTGGACTGAGCTTGGCCGGACTGAGCGAGTACGGAACTGTTCAAACCCTGAGTTCTCCAAGACTCTGCAGCTTGAGTACCACTTTGAAACAGTCCAAAAGCTCCGCTTTGGCATCTATGACATAGATAACAAGACACCTGAGCTGGCGGACGATGACTTCCTAGGAGGAGCTGAGTGTTCCCTGGGTCAGATTGTGTCCAGCCAGACAATAACTCTACCCTTGATGCTGAAGGCTGGAAAACCTGCTGGGCAGGGAACCATCATGGTCTCAGCTCAAGAGCTAAAGGATAATCGTGTAATAACCATGGAGGTGGAGGCCAGAAACCTAGATAAGAAGGACTTCCTAGGAAAATCGGACCCATTCCTGGAGTTCTTCCGCCAAGGTGACGGGAAATGGCACCTTACCTACAGATCTGAGGTAATCAAGAACAACCTGAACCCTACATGGAAGTGCTTCTCAGTCCCCCTTCAGCACTTCTGTGGGGGAGATCTCAGCACACCCATCCAAGTACGATGCTCAGACTATGATAGTGATGGCTCACATGACCTCATTGGCACTTTCCACACTACCGTGGCCCAGCTCCAGTCAGTCCCAGCCAACTTTGAATGCATTCATCCTGAGAAGCAGCAGAAGAAGAAAAACTACAAAAACTCAGGAATTGTCTGCATCAAGACTTGCCAGATAGAAACGGAGTATTCCTTCTTGGACTATGTGATGGGAGGCTGTCAAATCAACTTCACTGTAGGTGTGGACTTCACTGGTTCAAATGGAGACCCCTCTTCACCTGATTCCCTGCACTACCTGAGTCCAACGGGTGTCAATGAGTACCTTACAGCACTGTGGAGCGTGGGTAGTGTGGTTCAGGACTATGACTCGGACAAGCTGTTCCCAGCATTTGGATTTGGGGCTCAGGTGCCCCCTGACTGGCAGGTCTCCCATGAATTTGCCTTGAACTTCAACCCCAGTAACCCCTATTGTGCAGGGATCCAGGGCATTGTGGATGCCTACCGCCAAGCTCTGCCCCAAGTTCGCCTTTATGGCCCTACCAACTTTGCACCCATCATCAACCATGTGGCCAGGTTTGCAGCTCAGGCTGCACATCAAAGGACTGCCTCGCAATACTTCGTGCTGCTGCTGCTGACTGACGGTGCTGTGACCGACGTGGAGGCCACACATGAGGCTGTGGTGCGTGCCTCAAACCTGCCCATGTCAGTGATCATCGTGGGTGTGGGTGGCGCTGATTTCGAGGCCATGGAGCAGCTGGATGCTGATGGTGGGCCCCTGCGCACACGCTCTGGGGAGGCAGCTGTCCGTGACATAGTACAGTTTGTGCCCTACCACCGCTTCCAGAATGCCCCTCGGGAAGCATTGGCGCAGACTGTGCTTGCAGAAGTGCCTACGCAACTGGTCTCTTATTTTAAGGCCCAAGGTTGGGCCCCACTCAAGCCGCTTCCACTCCCAGCAAAGGGGCCTTCACAGGTCCCCCAGGCCTAG</CDS>GTTCCCTTGGAGGGTGGACTGTGGATAGTCATCAATCCTTTGTCCCCAGTGGCCTTTTGGGTCATGGCCCAACTCTGTATACATTCCCCAGTGCTGCTAGCACTTTATGTTTTATACTTTTGTACTTGCCCCAACTGCTGCTGCTCTTTTCACAACTTCTTGTTCCTAGCCTGGTGCCTCTCATTCAATAAAGACCAATGAAGACCA</gene></seq><eos>"
seq4 = "RNA, Odobenus rosmarus divergens, {}<seq><gene>GCCTGGAACATCCCACAGG<CDS>ATGGTGAGCGTCTGGACAATCGCGATTTTTCTTCTGGGAGCAGTCGAAGCAAAGGAAGTTTGCTATGAACAAATCGGGTGCTTTTCTGACTCGGAGCCCTGGGCGGGGACTGCAATCAGGCCCCTGAAAGTTCTCCCCTGGAGCCCAGAGAAAATCGGCACCCGCTTCCTGCTCTACACCAACAAGAACCCAAACAACTTTCAGACTCTCCTTCCCTCTGATCCATCAACGATTGAGGCGTCAAATTTTCAAACAGACAAGAAGACCCGGGTCATCATCCATGGCTTCATAGACAAGGGAGAAGAGGCCTGGCTGCTGAACATGTGCCAGAACATGTTCAAGGTCGAGGAGGTGAACTGCATCTGCGTGGACTGGAAGAAAGGTTCCCAAACCACGTACACACAGGCCGCCAACAACGTGCGAGTGGTGGGCGCGCAGGTGGCCCAGTTGCTCAGCATGCTCTCGGCAAACTACAGCTACTCACCTTCCCAAGTCCACCTCATCGGCCACAGCCTGGGAGCCCACGTGGCTGGAGAGGCAGGGAGCAGGACTCCAGGCCTGGGCAGGATTACAGGGTTGGATCCTGTAGAAGCAAGTTTCCAGGGTACTCCTGAAGAGGTTCGACTCGATCCCACCGATGCTGATTTTGTTGATGTGATTCATACAGATGCAGCTCCCCTGATCCCATTCCTGGGTTTTGGAACAAGCCAACTGTTGGGTCACCTTGACTTCTTCCCCAATGGAGGAGAGGAAATGCCAGGATGTGCGAAGAACCCCCTGTCTCAGATCGTGGACCTAGATGGCATTTGGGAAGGAACTCGGGACTTTGTGGCTTGCAATCACCTGAGAAGTTACAAGTATTACTCAGAGAGCATCCTCAACCCTGTTGGGTTTGCTTCATACCCCTGCGCTAACTACAGGGCCTTTGAGGCTAACAAGTGCTTCCCCTGCCCAGATCAAGGGTGCCCCCAGATGGGCCACTACGCTGATAAATTTGCCAGCAAGACAAGTGGTGAGCCGCAGAAATTCTTCCTGAACACAGGAGACTCCAGCAATTTTGCTCGCTGGAGATACGGGGTTTCTCTAACACTGTCTGGAAGAACAGCCACCGGTCAGGTCAAAGTTGCTTTGTTTGGAAGTAAGGGAAATACTCATCAATTTGATGTCTTCAGGGGGATTCTCAAACCAGGCTCTACCCATTCCAATGAGTTCGATGCAAAGCTTGATGTTGGAACAATTGAAAAAGTCAAGTTCCTCTGGAATAACAACGTGGTAAACCCAACCTTCCCCAGAGTGGGCGCAGCCAAGATCACCGTGCAAAAGGGAGATGATAAAACAGTGTACAACTTCTGCAGCGAAAGCACTGTGAAGGAGGACGTTCTGCTCACCCTCATGCCCTGTTAA</CDS>TGTCCAGGCGCCATCCGGCCACTGTGTTAACAGCAATAAAAACCACTGATGCATCGACCCGCTCCCAACCTCTGTCACTGATGTCAGCTGCCCCTGTGGGGACATTTGGGGGTAGCTCATTTAAATAAGTCTTGAGTGTTTCCAGATGTTTGGG</gene></seq><eos>"
seq5 = "RNA, Manis pentadactyla, {}<seq><gene>GGCTGCCATCTTGCCCGGTGCGGAGCCGCTGTCGCTCCCCCGAAAGCAGGGCGGAAGCAAGTGTGTCCTTCTCGAATTGCGTCCCTTTCCACTGCGAGACCTATCCGACAGCCTCCGCTGGCTGCCGAGTCCGAGTCTAGGCCAATGAACTAACCCACCACCCGCTCTCCACCCCGCCGAGGCACCCAGCCTCCCGCCGGCCGCG<CDS>ATGCCGAACGTGCAGCTACCGCCCAAGGAGAGCAACCTCTTCAAACGCATTCTGAAATGTTATGAACAGAAGCAATACAAAAATGGCCTCAAGTTTTGCAAGATGATTCTTTCAAATCCAAAATTTGCTGAACATGGAGAAACTTTGGCTATGAAAGGATTAACATTGAACTGTTTAGGAAAAAAAGAAGCAGCATATGAATTTGTTCGTAAAGGACTTCGTAATGATGTCAAGAGTCATGTCTGTTGGCATGTATATGGACTCTTGCAGCGTTCTGATAAGAAGTATGATGAAGCTATTAAATGTTACCGAAATGCTCTCAAATTAGATAAGGATAATATGCAAATTTTGAGGGATCTGTCTCTGTTGCAGATTCAAATGAGAGACCTTGAAGGTTACCGAGAGACAAGGTATCAGCTTCTTCAGCTGCGCCCAACACAACGTGCTTCCTGGATTGGATATGCTATTGCATACCATTTGCTGAAAGACTATGATATGGCATTAAAACTGTTGGAAGAATTTAGACAAACTCAGCAAGTTCCTCCCAATAAAATAGATTATGAATATAGTGAACTGATCTTATACCAGAATCAAGTGATGAGAGAGGCAGATCTATTTCAGGAATCTTTGGATCATATAGAAACTTATGAGAAGCAAATATGCGATAAACTTTTGGTGGAAGAAATTAAAGGGGAAATGCTGTTGAAATTGGGAAGATTAAAAGAAGCCAGTGAAGTATTCAAAAACTTGATTGATCGGAATGCAGAAAATTGGTGTTATTATGAAGGCTTGGAAAAAGCTCTACAACTTAGCACTTTAGAAGAGAGGCTTCAAATTTATGAAGACATTAGTAAGCAGCACCCCAGAGCAATTTCACCTAGAAGATTACCTTTGAATCTTGTCCCAGGTGAAAAATTTAGAAAGCTAATGGATAAGTTCCTGAGAGCTAACTTCAGTAAAGGCTGCCCACCCTTATTTACCACTTTGAAATCTTTATTTCACAATGCAGAAAAGAAAATGGGGAGAAGGAACCCCCAACGACACTACTCTGGGTTCAGTATTTCCTGGCACAGCACTTTGATAAACTTGGACAATATTCTTTGGCTTTGGATTATATCAATGCTGCGATTGCTAGTACTCCAACTCTGA</CDS>TAGAATTATTCTATATGAAAGCAAAAATTTACAAGCATATAGGTAATCTCAAAGAAGCTGCAAAGTGGATGGATGAAGCACAGTCTTTGGATACAGCTGATAGATTCGTCAATTCCAAATGTGCAAAATATATGCTTCGAGCAAATATGATAAAAGAAGCAGAGGAAATGTGCTCCAAAT</gene></seq><eos>"
seq6 = "RNA, Geospiza fortis, {}<seq><gene><CDS>ATGAATGTACCAGACCTCAGAGTGCTGTGGCTGGCCTGGAGGAGGGAGGTGGAGCTGCTCTCCGATGTGGCTTATTTCGTGCTCACCACGCTGTCAGGCTATCAGACCCTGGGAGAAGAGTATGTGAACATCATTCAGGTTGACCCGAGCAAGAAGAAGGTGCCTTCCTTCCTTCGTCGGGCCCTCTTCATTTCCCTGCACACCTTGGTGCCCTACTGCCTGGAAAAGGCACTGCTGCACCTGCAACACGAGCTGCAGGTGGAGGCTGAGGAATCCAGGAGCCCAGAGAACCCCCCAGCACTTGGCTTTCCCAGCAGGAGCTTCATCCGCAGCTGGGTTCGGAAGCAGGTTGGGCAGCTCAGTGAGCAGCAGAAGAAAAGAACCTCCCAAGTCGTGGATGTCCTCAAACAATCCATCCCCCTGCTGCACCGGCTCCACCTGGCAGTGTTCTACATCCAGGGCACCTTCTATCACCTGTCCAAAAGGGTCACAGGCATCTCATATCTGCATTTTGGAGGACTGCAAGGAGAAGATCAGAGTATCAGATCAAGTTACAAGTTTCTTGGAATAATTTCTCTCTTCCACCTTCTGCTGACCATTGGAGTTCAGATGTACAGCTTCAAACAGAAGCAGAGAGCAAGGCAGGAGTGGAGGCTGCACCGCAACCTCACCCACCAGAAAAGCAGGAGCAAGGAGGCGGCGGCCGGGCGCCAGTCGCGCTGCACGCTGTGCCTGGAGGAGAGGAGGCACAGCACAGCCACCCCCTGTGGGCACCTCTTCTGCTGGGAGTGCATCACTGCCTGGTGCAGCACCAGGGCAGAATGTCCACTGTGCAGAGAGAAGTTCCATCCTCAGAAACTGATCTACCTACGGCACTACCAAATGTAA</CDS>AGGGAACCTCTTCTGTCTCCTCATACTTGGGTTGGTCCAAGAAAGGCTTTATTAAACTGGAAACTGCCTGTGCAAGGTGGTACTACATTCCTGTGTGACAGAATAAAGCTATTTTTGCTTCTATATTCAA</gene></seq><eos>"
seq7 = "RNA, Perca flavescens, {}<seq><gene>CCGTCGCCATCCTTGGAAAACCCAAATTTGCGGCACATTCGTGTTGTTTACTGGGCTAGTGACACCAAGGCAATGCTCCAAGGAGGTTTACGATTCTGAGCGCCTGCTGACAGCCTTGACAGAGATACTTTTAACTGAAGGAGGAAGGC<CDS>ATGCTGCCTTTAGTTGGAGCTGAGTTGACACCAGTGCAGCTCTATAGACATCTCCTCCGATGTTGCAGGCAGTTACCTACCCCAGCAATGCAGCAGCATTATCGACATGCCATAAGACAGGGTTACATCAGTCATTCCGATGAAGACGACCAAGAGAGGATACAAATGATAATCCAAAGAGCTATTGCAGATGCAGACTGGATTCTTGATAAATATACCAAGAAGAAGTGA</CDS>AGTTCATAACCCATGAATACGTCCTCGGGTGTAGCATTTTGCAGAGCTGCACCTGACATGAATCGTAGTGGTGGATGCACCGTTTCTATGTCTGACTTTTGAGTGCTCATTGTGTTATTTGTCCTCATCATTGTGGGGATGTGAACAGCCATGGTCAAGTCAAGTCAGCTTTATTGTCATATGTGCTATACGTGCATGACATACGGCACAAATGAAATTTCAGTCCTTTCGGACCCCCGGTGCAAAGCTACATGTGATGAGAGGAAGACCTCACGAGTCATTGATGATATACTATACTGAATGATATGACTGCTCAATTGAATATAATATTCACTATCACTAAAAGTGTAAATACTGTTCTTAGATTGAAATGAGTTGACTATATACTGAATTCAATAAATTATTGTCAATATTTATA</gene></seq><eos>"
seq8 = "RNA, Carlito syrichta, {}<seq><gene><CDS>ATGCGACCAATAGCGCGGGGCAGGAGGCGGGTTCCGGCTATCCATGGCTTCCGGGAGACAGAGAAGTCTCGCTGGTCAGAGGCCAGCCGGACCATCCTGCAGCGCGTGCAGGCAGCCGCCTTCAGCCCCGGCCAGGCCCTGCTCTCCCCGGTGCACGTGCTGGACCTGGAAGCTCAGGGCTACATCAAACCCCACGTGGACAGCATCAAGTTCTGCGGAGCCACCATCGCCGGCCTGTCGCTCCTGTCTCCCAGCATCATGCGGCTGGTGCACACCCAGGAGCCGGGGGAGTGGCTGGAACTCCTGCTGGAACCAGGCTCCCTGTACATCCTTAGGGACTCGGCCCGCTATGACTTCACCCATGAGATCCTTCGGGACGAAGAGTCCTTTTTCGGGGAGCGTCAGATCCCCCGGGGCCGACGCATCTCCGTGATCTGCCGCTCCCTTCCTGAAGGGATGGGGCCAGGGGAGCCCGGACAGCCCCCCCTGGCCTGCTGA</CDS>CTCCCGAGGCCTGCCCCCCAGCCTCCCACAGACACTGGATTTGTGAATAAAGTTGAAGAATGAACA</gene></seq><eos>"
seq9 = "RNA, Poecilia reticulata, {}<seq><gene>AACGCACCCCGCTGCTGTCAGAACGCCCTGTAGCCGAGTCTCACAGAGCTCAAGTCCTTCCAGGGGTTCCCCATTCACAAAGCCCGCCCGGCTGCCTGACATTATTACGGAGGGCGGAGGGAGGAAGAGGAGAAGGGCGGGAGGGAGGAGAGAGAACGAGAGTTGAGACGTCCCTTCCTCACTTGGCTGCATTCGGTTTACCGACGCCGGAGAAGCGACC<CDS>ATGCTGTGGAAACTAGTTGAGAATGTCAAGTATGAAGATATTTACGAGGACCGGCATGACGGTGTCTCAAGCCACAGTTCGCGCCTGTCCCAGCTGGGCTCGGTGTCGCACGCCGGACCTTACTCCAGCGCGCCGCCGCTGTCTCATGCGCCTTCCTCGGACTTCCAGCCGCCATACTTCCCTCCGCCATACCAGCCGCTCGCTCACTACCAGAGCCAGGACCCGTACTCCCACGTCAGCGACCCGTACTCCCTGAACTCGCTCCATCAGAGCCAGCAGGGCGCATGGGGCGCGCGTCAGCGGCAGGACGCCGCCGGGGACCGGATGGAGAGCTCGGCTTTGCTGGCGCAGCCTCGGGCCTCCCTGCCGCAGCTTTCCGGGCTGGACCCACGGCGGGATTACGGCGGCGTGCGGCGGCCTGACGTGCTGCTGCACTCCGCACATCCGGGACTGGAGCCCGGCATGGGAGACGGACTGCTGCACGGTCTGCACGGCATGGAGGACGTTCAGACCATCGAAGACGCCAACGGAACCAATATCCTGGATCAATCGGTAATTAAGAAAGTCCCCATGCCCCCCAAGAACGTGGGCTCCCTGATGCTCGGCAAGGACGGCCTGATCGGCGGAGTCACCGTCAACATTAATGAGGTGTTCTGCTCGGTACCGGGCCGCCTGTCGCTGCTCAGCTCCACCTCCAAGTACAAAGTGACCGTAGGGGAGGTGCAGAGGAGGCTGTCTCCACCTGAGTGCCTCAACGCCTCTCTGCTAGGAGGCGTGCTGAGAAGAGCAAAGTCTAAAAACGGTGGCAAATGCCTAAGAGAAAAGCTGGAGAAGATCGGATTGAATCTTCCAGCCGGGAGACGTAAAGCGGCCAACGTCACGTTACTAACATCTCTTGTAGAAGGTGAGGCGGTGCACCTGGCCCGGGACTTCGGCTACATCTGCGAGACGGAGTTTCCCACGAAAGCTGTGAGCGAGTACCTGAACCGGCAACATGCGGACCCCAACGAGCTGCACACGCGGAAAAACATGCTGCTGGCCACAAAACAACTGTGTAAGGAGTTCACGGACCTGCTGGCCCAGGACAGGACTCCGCTGGGGAACTCCAGGCCGTCCCCCATCCTGGAACCCGGCATCCAGAGCTGCCTATCCCACTTCTCCTTCATCACACACGGCTTCGGCTCGCCCGCCATCTGCGCCGCGCTCACCGCCCTGCAGAACTACCTCACCGAGGCTCTCAAAGGACTCGACAAGATGTTCCTCAACAACCCGCCCAACAGCCGGCATGGGGACGCGGGCGGCAAGGCCGGCGACAAAGAGGAAAAACAGCGGAAATGA</CDS>AGCCGGCGCAGAAAGGGGGAGTGATGGAGGAGAGGCGGGAGGGAGGCCGACACGCTGACTGAGGCGTTACACCGTTTCTCGTAGCTTCTACAGCGCCGCCACCGCCACCCTGCCACGCACACGGCGGAGGTGGAGGACAGGGGAGGCTTGGCGTGGAGGAAGACACCCGTGGGCCAGTGTCTGAACGCACGGAGGGATTTTTCATCTGTGTTACGTTCTTCTCATTCCCCCTGAGGACATGGTGGTTCCTGTGTTCCATCCATTCTGCTCCCCCCGCCTGCATCCTGTGAGACTGAGATGCTGCGAGCAAACGACATCGTGTCCGTGTGCCTGAGTGTGTGTGAGAGAGAGACTGTGAGAGAGTGTGTGTGCGCTTGTGTTTGTCTTCACACCGAAACAGAAGAAAAAAGAAAAAAAAAGCGTTGTGGGTTTGGAGATCCATTTTTTTGTTATTTTTTTGTGCTGCAGCATCTATCTGTGAAACAGGCTGGGTATTTAACGAGACTCCAACGAGGGACGGCGTTAATTTATGTGTGTAAATATTAATTTATAATTATTTAAGTTGATCAAATAGGTGCGCCAGCTGCTTTGGAGTCGACCTATTTATGTGTGATTTGACCTTTGTGAGCCAGTGGGCCGGACTCGTTCAGTGTTTTTCTGTTCTTATAGAGCTGTTGTCATGGTGATGATGATGATGATGATGATGGGTAGACATGCTTCTAATTGTCCAGTGATTTCATTTCTATCATATTGATAATAAACCATGTGTTTTATAATGTGA</gene></seq><eos>"

annos = ["SPRY domain-containing SOCS box protein 3 isoform X5", "relaxin-3 receptor 1", "copine-1 isoform X1",
        "inactive pancreatic lipase-related protein 1", "N-alpha-acetyltransferase 16, NatA auxiliary subunit isoform X7", 
        "peroxisome biogenesis factor 10", "LYR motif-containing protein 9", 
        "alpha-ketoglutarate-dependent dioxygenase alkB homolog 7, mitochondrial", "transcription factor AP-2-beta isoform X1"]

syms = ["SPSB3", "RXFP3", "CPNE1", "PNLIPRP1", "NAA16", "PEX10", "LYRM9", "ALKBH7", "TFAP2B"]

seqs = [seq1, seq2, seq3, seq4, seq5, seq6, seq7, seq8, seq9]


In [ ]:
results = []
for i, (seq, origin_sym) in enumerate(zip(seqs, syms)):
    for j, anno in enumerate(annos):
        replace_sym = syms[j]
        seq_filled = seq.format(anno)
        seq_tokens = tokenizer.encode(seq_filled)
        res = sequence_likelihood_no_bos_short(model, seq_tokens, device=device)

        results.append({
            "origin":   origin_sym,
            "replace":  replace_sym,
            "perplexity":   res["perplexity"],
        })
df = pd.DataFrame(results)


In [ ]:
mat_perplex = df.pivot(index="origin", columns="replace", values="perplexity")
df['Type'] = np.where(df['origin'] == df['replace'], 'Matched', 'Mismatched')


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LogNorm
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(5.5, 4.5), dpi=300)

heatmap = sns.heatmap(
    mat_perplex,
    ax=ax,
    cmap="viridis",
    norm=LogNorm(vmin=mat_perplex.min().min(), vmax=mat_perplex.max().max()),
    annot=True,            
    fmt=".0f",             
    annot_kws={"size": 9.5, "weight": "normal"}, 
    linewidths=0.5,
    linecolor='white',
    cbar_kws={"shrink": 0.8}
)

cbar = heatmap.collections[0].colorbar
cbar.set_label("Perplexity (Log Scale)", size=12)
cbar.ax.tick_params(labelsize=10)

for i in range(mat_perplex.shape[0]):
    ax.add_patch(patches.Rectangle((i, i), 1, 1, fill=False, edgecolor='red', lw=1.5, clip_on=False))

ax.set_xlabel("Provided Gene Symbol (Prompt)", fontsize=12)
ax.set_ylabel("Real Gene Symbol", fontsize=12)
ax.tick_params(axis='both', which='major', labelsize=10)

ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl

mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Helvetica']
mpl.rcParams['xtick.direction'] = 'out'
mpl.rcParams['ytick.direction'] = 'out'
mpl.rcParams['axes.linewidth'] = 1.2
mpl.rcParams['xtick.major.width'] = 1.2
mpl.rcParams['ytick.major.width'] = 1.2

fig, ax = plt.subplots(figsize=(3, 4.5), dpi=300) # 稍微调整比例

box_palette = ['#D6604D', '#00897B'] 

sns.boxplot(
    x='Type', y='perplexity', data=df, 
    palette=box_palette, 
    width=0.6, 
    showfliers=False, 
    linewidth=1.2, # 统一线条粗细
    ax=ax
)

sns.stripplot(
    x='Type', y='perplexity', data=df, 
    color='black', alpha=0.3, size=4, jitter=0.2, ax=ax
)

ax.set_yscale('log') 
ax.tick_params(labelsize=12)

ax.set_ylabel('Perplexity (Log Scale)', fontsize=14)
ax.set_xlabel('', fontsize=14)

sns.despine() # 移除 top 和 right 边框

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Helvetica']
mpl.rcParams['axes.linewidth'] = 1.2
mpl.rcParams['xtick.major.width'] = 1.2
mpl.rcParams['ytick.major.width'] = 1.2


In [ ]:
seq1 = "RNA, Xenopus tropicalis, {}<seq><gene>AGCATCTGGTGTGGACTGCTTATTGCACAACACAAACAACA<CDS>ATGCATCTTACAGCTGATGACAAGAAACACATCAAGGCCATTTGGCCTTCTGTAGCTGCTCATGGTGACAAATATGGCGGAGAAGCTTTGCACAGGATGTTCATGTGTGCTCCCAAGACCAAAACCTACTTTCCTGATTTTGACTTCAGCGAACATTCAAAACACATCTTGGCTCATGGCAAGAAAGTTTCGGATGCTCTGAATGAGGCTTGCAACCATCTGGACAACATTGCCGGATGCCTGTCCAAGCTGAGTGACCTCCATGCCTATGACCTGAGAGTGGATCCAGGCAACTTCCCATTGCTGGCCCATCAAATTCTGGTGGTTGTTGCTATCCATTTCCCTAAGCAGTTTGACCCTGCAACCCATAAGGCCCTGGACAAGTTCCTGGTTTCCGTATCTAATGTTCTGACATCCAAATATCGTTAA</CDS>GGCTCAGCAGTAACAGTAGCAGAAGTTTGGACATCAGACATCAGTTAATGACAAACAATCAAACTGACACAGCTTGTGAAAGAATGTTCTGAAATAAACATTTTTAAAATT</gene></seq><eos>"
seq2 = "RNA, Oncorhynchus mykiss, {}<seq><gene>ACACAAGACATTAATTTGTTTAATCTTGTTGCAAAACCGGAAAACAAACACTGACCAC<CDS>ATGGTTGAATGGACAGACGCAGAGAAGAGCACCATCAGTGCTGTCTGGGGCAAAGTAGATATCAATGAGATCGGACCACTGGCTCTGGGAAGAGTCCTGATCGTCTACCCCTGGACTCAGCGTTATTTCGGCTCTTTCGGAGATGTGTCCACTCCCGCAGCAATCATGGGCAACCCAAAAGTCGCTGCTCACGGCAAGGTCGCGTGTGGAGCTCTGGATAAAGCTGTGAAGAACATGGGCAACATCTTGGGCACATACAAGTCACTGAGCGTGACCCACGCTAACAAACTCTTCGTCGACCCTGACAATTTCAGGGTGTTGGCTGACGTCCTCACAATTGTCATTGCCGCCAAGTTCGGAGCCTCTTTCACTCCTGAAATTCAGGCAACCTGGCAGAAGTTCATGAAAGTGGTTGTCGCAGCCATGGGCAGTCGGTACTTCTAA</CDS>ATGCACCACAGACAAGGGATTATGTTCTGACATCCCCGCCTGAGTAAAAATAAGGACATTGGAACAAAAAAAA</gene></seq><eos>"
seq3 = "RNA, Ficedula albicollis, {}<seq><gene>CCAGCTACCCTTTGGGGAAACACAAACGGCAGCAGGAAGCACTCGGGACCTGTCTTTGGTGAACC<CDS>ATGGGGCTCAGTGACCAGGAATGGCAGCAAGTCCTGACTGTCTGGGGAAAGGTGGAGTCCGACATCGCTGGCCATGGCCATGAAGTTTTAATGAGACTCTTTCAGGACCATCCTGAGACCCTTGATCGCTTTGAGAAGTTCAAAGGCCTGAAGACCCCTGATGCGATGAAGGGCTCTGAAGACCTGAAGAAACATGGAACTACTGTTCTTACCCAGCTGGGCAAAATCCTGAAGGCGAAGGGTAATCATGAGGCTGAGTTGAAGCCTCTGGCTCAGACCCACGCAACCAAGCACAAAATCCCTGTCAAATATCTGGAGTTCATTTCTGAAGTCATTATCAAGGTCATTGCCGAAAAACACGCTGCAGACTTTGGGGCTGATGCCCAGGCTGCAATGAAGAAGGCTCTGGAGCTGTTCCGAAATGACATGGCCAGCAAGTACAAGGAGTTCGGTTTCCAGGGTTAG</CDS>CACATACGCACAAAGGAACACCACGATGGAGCACCTGGCCACGCATCTCTCCCCAGCGCTATTTTGGACATCTCACAATTGGCCATCTCTGATGTGTGGGAAAGATTTGAGGAGAGGCCAAGAGTCACTCCAGGCAGCATAGAGGCACCCAGAGTGATATCCATGATAAACTGTCATTTCCTGGCTTCGTGTATACAAAAGAAATATCCTGCTTTCTTAGGAAAAATAACATTAGGGGTTATCTTTGAGCTTCAGCCTATATTATCCCCTTTCTCTCTCTCTTCCTGAAACTCTGTCTCATCTGAGAGGCAAATGGCATGGCCAAAGATGAAGGGATGTGTAGAATCACGAGCAGTGTTGAGTGCGGTGAATGAGTGAAGGAATGACTCAAATGAGAGATGTGAGAATGGGAAGCAGGGACACAGCATCCCATTCCTCCCATAGGTGAGACTGTTGCAGGTTACTATGGAGAACTTCCAGCACAGTGTTACCCTGCTTTGTGGGAGTGAATGCAGCCAGTCAGTCTCCAGTGCCTGCCAGCTCTGGAGGGTTTAGTTGTTAAACTTGCATTATAAAGATCGGTACTCCCTGTGATCTGTAAGCTTCTAGCAGTTGCCACTATGCTATGTCTGAAGGTCTGTTGTACTGGGAAGTGAAGTATGTGTTTTTCAGAAAATAAAAATCCCTGCACCAGA</gene></seq><eos>"
seq4 = "RNA, Scomber scombrus, {}<seq><gene><CDS>ATGTCTCGCAGGGAGTCTCCACCGCCGCCGTCGCCACCTCCAACCCCGCATCTGCTCGGACTGAGGAGAGCGGAGGTGGAGTGCGACGATCGTCCGGAGCGGGCCGAGCCGCTGTCAGATGCAGAGAGAGAGATCATCCAGGACACGTGGGGGCACGTCTACAAGAACTGCGAGGACGTGGGGGTGTCCGTGCTCATAAGGTTCTTTGTGAACTTCCCGTCAGCCAAACAGTACTTCAGCCAGTTCCAGGACATGGAGGATCCCGAGGAGATGGAGCGAAGTAGCCAACTTCGGCACCATGCCCGCAGGGTAATGAACGCCATCAATACTGTGGTAGAGAACCTCCATGACCCAGAGAAGGTGTCGTCAGTACTGGCCTTGGTGGGAAAGGCCCATGCTATCAAACACAAAGTGGAACCCATGTACTTCAAGATCCTGAGCGGCGTGATGCTGGAGGTATTGTCTGAAGATTTCCCAGAATTCTTCACGGCGGAGGTGCAGATGGTCTGGACCAAATTAATGGGGGCGGTGTACTGGCATGTGACCGGGGCCTACACAGAGGCGGGCTGGCTCCAGGTCTCCAGCTCAGCGGTGTGA</CDS></gene></seq><eos>"
seq5 = "RNA, Capricornis sumatraensis, {}<seq><gene>AGGGCCAATCTGATAGTGGTGCTGTTTATAAAAGTCCCCTCAGCTGGGGCCAGAAGCTCTGGATCAGCCCCCACAATCCTGCTGACCTCCAGCACTTGCCTCTACAGTGGGTACAG<CDS>ATGGCACTGTCCCACCCCAGTCCCTTCTCAGCCATGGAGCTTCTCCTGGCCTCTGCCGTCTTCTGCCTGGTATTCTGGGTGGTCAGGACCTGGCGGCCTCGGGTCCCTCAAGGCCTGAAGAGTCCACCAGAGCCCTGGGGCTGGCCCCTGCTCGGGCACGTGCTGACCTTGGGGAAGAACCCACACGTGGTCCTGTCGCAGCTGAGCCAGCGCTATGGGGACGTGCTGCAGATCCGCATTGGCCGCACACCCGTGCTGGTGCTCAGCGGCCTGGACACCATCCGGCAGGCGCTGGTGCGGCAGGGCGATGATTTCAAGGGCCGGCCCGACCTCTACAGCTTCACCTTGGTCACTGACGGCCAGAGCATGACCTTCAACCCAGACTCTGGACCGGTGTGGGCTGCCCGGCGACGCCTGGCCCAGAATGCTCTCAACACCTTCTCCGTGGCCTCAGACCCATCTTCCTCCTCCTCCTGCTACCTGGAGGATCATGTGAGCAAGGAGGCTGAGGCCCTCCTGGGCAAGTTCCAGGAGCTGATGTCAGGGCCTGGGCGCTTCGACCCCTATGGCCACGTGGTGGCATCGGTGGCCAACGTCATCGGTGCCATGTGCTTCGGGCAGCACTTCCCCCAGAGCAGTAAGGAGATGCTCAGTGTGGTGGAGAGCAGCCATGATTTCGTGGAATCCGCCAGCTCTGGGAATCCTGTGGACTTCTTCCCCATCCTTAAATACCTGCCCAACCCGGGTCTGCAAAGGTTCAAGAGCTTCAACCAGAGGTTCCTGCAGTTCGTGCGGAAAACAGTCCAGGAGCACTACCAGGACTTTGACAAGAACAGCGTCCAGGACATCATAGGCGCCCTGTTCAAGCACAGTGAGGATAACTCCCAAGCCAGCAGTCGCCTCATCTCCCAGGAGAAGACTGTCAACCTTGTTAACGACCTCTTTGGGGCCGGGTTTGACACCATCACAACAGCCATCTCCTGGAGCCTTATGTACCTTGTGATAAGTCCTAAGATACAGAGAAAGATCCAGGAGGAGCTGGACAGAGTGATTGGCAGGGCACGGCGGCCCCGGCTTTCCGACAGATCCCAGCTGCCCTATTTGGAGGCCTTCATCCTGGAGACCTTCCGACACTCCTCGTTCGTCCCCTTCACCATCCCACACAGCACAACAAGGGACACAACACTGAATGGCTTCTTCATACCCAAGGAACGCTGTGTCTTCATAAACCAGTGGCAGGTCAATCATGACCCGAAGCTGTGGGGGGACCCATCTGAGTTCCGGCCAGAGAGATTCCTCACGTCTGATGGCACCGCCATTGATAAGTTCGCAAGCGAGAAAGTGTTACTCTTCAGCATGGGCAAGCGCCGGTGCATAGGGGAGGTCATGGCCAGGTGGGAGGTCTTCCTCTTCCTGGCCATCTTGCTGCAGCGGCTGGAGTTCAGCGTGCCAGCAGGTGTGAAAGTGGACCTAACCCCCATCTACGGGCTGACCATGAAGCACCCCCGCTGTGAGCATGTGCAGGCACGGCTGCGCTTCCCCACCAAGTGA</CDS>AGAAGTTGCTGGCGTCTGAGGCAGAGGGAAGGGAAGGGTGGCGGTCATTGAGGAAGCCTCATTTCTTTTCCTTCCTTTCTTTTTAACTAACAGCTATATATACCATACAATTGTAGGCCAACTATACTTCAAAAAATAAACAACCTCATAGAAACAGAGATTAGATTTGTGATTACAGTGAGAGGGAAAATTTGAATGAAGGTGGTCAAAAGGCACAAATTTCCAGTTATGAGATAAATAAGTACTAGAAATGTGATATATAATGTGATTAATATAATTAACACTACTGTACATTATATATGAAAGTTGTTGAGAGTAAATCCTAAAAGTTCTCATCACAAGAAATAATTTTTTTCTATT</gene></seq><eos>"
seq6 = "RNA, Chlorocebus sabaeus, {}<seq><gene>GCTGTATTAATAAGAAGAGAAGTCTTCA<CDS>ATGGATCCTTTTGTGGTCCTGGTGCTCTGTCTCTCCTTTGTGCTTCTCTTTTCACTCTGGAGACAGAGCTCTGGGAGAAGGAACCTCCCTCCTGGCCCCACTCCATTTCCTATTATTGGAAATATGCTACAGATAGATGTTAAGGACATCTGCAAATCTTTCAGCAATTTCTCAAAAGTCTATGGTCCTGTGTTCACCGTGTATCTTGGCATGAATCCTGTAGTGGTGTTGCATGGATATGAGGCAGTGAAGGAAGCCCTGATTGATAATGCAGAGGAGTTTTCTGGAAGAGGCATTTTGCCAATATCTGAAAGAATTACTAAAGGACTTGGAATCATTTCCAGCAATGGAAAGAGATGGAAGGAGACCCGTCGTTTCTCTCTCACAACCTTGCGGAATTTTGGGATGGGGAAGAGGAGCATTGAGGACCGTGTTCAAGAGGAAGCCCGCTGCCTTGTGGAGGAGTTGAGAAAAACCAAGGCCTCACCCTGTGATCCCACTTTCATCCTGGGCTGTGCTCCCTGCAATGTGATCTGCTCCGTTGTTTTCCAGAAACGATTTGATTATAAAGATGAGAATTTTCTCACCCTGATGAAAAGATTCACTGAAAACTTCAGGATTCTGACCTCCCCATGGATCCAGGTCTGCAATAATTTCCCTCTACTCATTGATTGTTTCCCAGGAACTCACAACAAATTGCTTAAAAATGTTGCTCTTACAAAAAGTTACATTAGGGAGAAAGTAAAAGAACACCAAGCAACACTGGACATTAACAATCCTCGGGACTTTATCGATTGCTTCCTGATCAAAATGGAGAAGGAAAAGGACAACCAACAGTCAGAATTCACTATTGAAAACTTGGTTGGCACTGTAGCTGATCTATTTGTTGCTGGAACAGAGACAACAAGCACCACTCTGAGATATGGACTCCTGCTCCTGCTGAAGCACCCAGAGGTCACAGCTAAAGTCCAGGAGGAAATTGATCATGTAATTGGCAGACACAGGAGCCCCTGCATGCAGGATAGGAGCCACATGCCTTACACAGATGCTGTAGTACACGAGATCCAGAGATACATTGACCTTGTACCCACCGGTGTACCCCACGCAGTGACCACTGATATTAAGTTCAGAAACTACCTCATCCCCAAAGGCACAATCATAATGACATTACTCACTTCTGTGCTGCACGATGACAAAGAATTCCCTAATCCAAAGATCTTTGACCCTGGCCACTTTCTGGATGAGAATGGCAACTTTAAGAAAAGTGACTACTTCATGCCTTTCTCAGCAGGAAAACGAATTTGTGCAGGAGAAGGACTTGCCCGCATGGAGCTATTTTTATTTCTAACCACAATTTTACAGAACTTTAACCTGAAATCTGTTGCTGATTTAAAGAACCTCAATACTACCTCAGCTACCAGAGGGATTATTTCTCTGCCACCCTCGTACCAGATCTGCTTCATTCCTGTCTGA</CDS>AGAATGCTAGCCCATCTGGCTGCCGATCTGCTATCACCTGAAACTCTTTTTTATCAACGACATTCCCACTGTTATGTCTTCTCTGACCTCTCATCACATCTTCCCATGCACTCAATATCCCATGAGCATCCAACCTCCATTAAGGAGAGTTGTTCAGGTCACTGCACAAATATATCTGTAATTATTCATACTCGTAACACTTGTATTGACTGTCACATATGCTAATAGTTTTCTGATGCTGACTTTTTAATATGTTATCACTGTAAAACACAGAAAAGTGATTAATGAATGATAATTTAGATCCATTTCTTTTGTGAATGTGCTAAATAAAAAGTGTTATTCATTGCTGGTTCA</gene></seq><eos>"
seq7 = "RNA, Anas acuta, {}<seq><gene>TTTGGGACTGAGCCAAGTTCCACATTCACAAGCCAGAAGCTGCAGGGAGAGGGGAGCAGGGAGACAGAAAAGGAGAGTGAGGGGACAGGAGGAAGCAAGGAAAAGGGGACAGCTGCCTGGGGAGGGAGGCAGAACAGAGAAGGAGGCAGAAGGGAGAGGCAGAAACAGTCTTGGGTCAAAAGAGGAAGACCGAGGAGCAGGCAGGCTGCAAA<CDS>ATGACATTGCTCCTGTGGCTGTCATCTGACTGGAGCAGCATTGCTGTGCTGGGAGTTTTCCTTACCGTCTTCACTTTGGTTTTTGATTTCATGAAGCGCAGAAAGAAGTGGAGCCGGTACCCGCCAGGCCCCATGTCGCTGCCGTTCGTCGGGACCATGCCGTACATCGACTTCAACAACCCGCACCTCTCCTTCAACAAGTTTCACAAGAAGTTCGGAAACATCTTCAGCCTCCAGAACTGCTGGACCAACGTGGTGGTGCTGAACGGGTACAAAACCGTGAAGGAAGCCCTGGTCCACAAAGCAGATGACTTTGCTGACCGGCCGTACTTCCCAACGTATGAACATGTGGGCTACGGAGATAAATCTGAAGGGATTGTTGTAGCAAGATACGGGCATGTCTGGAAGGAGATAAGGAAATTCACACTCTCTACCTTGAGGGACTTTGGAATGGGAAAGAAATCCTTGGAGGAGCGAGTGACTGAGGAAGCAGGATTCCTGTGCTCTGCAATCAAAGCAGAAGAAGGTCGCCCTTTCGACTTACGTTTTCTTGTAAACAATGCAGTCTGCAACATGATCTGCAGCATCATCTATGGAGGGCGTTTTGACTATGGTGATGAGGTGTTCAGGAAAATGGTAGCTTTGTTTGAAAACTCCCTGAGTGAAGAAGCCGGATTCCTGACTCAGTTTCTTAATGTGATGCCCATTTTGTTGCGCATCCCTGGACTGCCGCAGAAAGTCTTTCGCGGGCAAAAGGCATACATGGACTTCATAGATGTCCTCATACAGAAGCACATGGAGACCTGGAACCCTGAACACACCCGAGACTTCACTGATGCGTTTTTAAAGGAGATGTCGAAGGGTAAGGAGGCTGAAGAGAATGGCTTCAATTACAACAACCTTCGTCTGGTGACCGCAGACTTGTTTGCAGCTGGTTCTGACACCACCTCCGCCACCCTCCGGTGGGCATTTCTCTACATGCTTTTGCACCCAGGAACACAGAGTAAAATCCACAAAGAGATTGATATGGTGATTGGAAGAGAGAGGTCACCCACCATGATGGACCAAGTGAACATGCCCTACACCAATGCCGTGATCCACGAGGCACAGCGCTGTGGGGACATCGCTCCTGTTGGGCTACCTCACATGGCATACCGGGACACCGAGCTTCAGGGTTACTTCATTCCCAAGGGGACGACAGTCATCACCAACTTGTCTTCCGTGCTGAAGGATGAGTCAGTCTGGAAGAAACCAAATGAGTTTTACCCCGAACACTTCCTGGACGTGAACGGGCAGTTTGTGAAACCGGAGGCCTTCATGCCTTTCTCAGCAGGTCGACGTGTCTGCCTGGGGGCACAGCTGGCCAGGATGGAGCTCTTCATCTTCTTCACCACTCTCCTGCAGCAATTCACCTTCATGCTCCCCGAGGATCAGCCCAGGCCACGGACAGATGGCTACTTTGCCCTCATAAAATGCCCACACCCATTCAAGCTGCGAGCTATCCCAAGATAA</CDS>CTGCACACCACTCCCCTTTCACTTAGAGACCACCACAAGTGCTGTCAGAAATGCAGCAGGTCTAGGCAGGATGATGACAAGTCCCACTTAAGACCACCTAAACCTGTGCAGATGTCCCTAGCAGTGTAAAGAACGCTGTAGCCCAGTATATAGCTCAGCTTAATTCCCATCCTGGATCGTGAATCACACAGCCATAGCACATTACAGGTGACGGTCACATTTGCAAGGACGTATATAACCAATAAACCAGTTCTGCTGTACTTTGGGCTCTCGTTGCAGAAACAGTTTACCGAGCACTTGGCTACTGAGAGAGTATTAGCCCTGGCTTATTGTGTACTGAAATACGAAGGTATCTGGGGTGAAAGATAACCCCGGGGTCCTGCCAATTCTAGGAGGAAAACCAAACAAACAGACAAACCTAGGAACAATTTGCGAAGAGCTCCTGGCCACCTCCCTTCCCTGCCATCTCCTATCCGAATAGCTCCTCCAAACAGCTTTTCTTCAGCGTTACATAAAAACAGCTCTCGGGCAGGCTTGGCACGGAGCCGTGGGGTTGCATTGCCTGGGGGTGAGCTGTACATTTCGGGGCTGGCTCTCTTTTTATTCCTGCGTTATTTCTGACCAACTGTTTCGCGAGGTTTGCAAAACGATTAAGGCTCGATGTTCAGACTCATCCTTAACAAAATAATTATGTAAAACAAATAAA</gene></seq><eos>"
seq8 = "RNA, Symphalangus syndactylus, {}<seq><gene>TCTCCTGGCTGTCAGCCTGGTGCTCCTCTGTCAATATGGAACCCATTCACATGGACTTTTTAAGAAGCTTGGAATTCCAGGGCCCACACCTCTGCCTTTTGTAGGAAATATTTGGTCCTACCGTAAGGGCTTTTGGACGTTTGAC<CDS>ATGGAATGTTATAAAAAGTATAGAAAAGTCTGGGGTTTTTATGACTGTCAACAGCCTGTGCTGGCTATCACAGATCCCGACATGATCAAAACAGTGCTAGTGAAAGAATGTTATTCTGTCTTCACAAACCGGAGGCCTTTCGGGCCAGTGGGATTTATGAAAAGTGCCATCTCTATAGCTGAGGATGAAGAATGGAAGAGAATACGATCATTGCTGACTCCATCTTTCACCAGTGGAAAACTCAAGGAGATGGTCCCTATCATTGCCCAGTATGGAGATGTGTTGGTGAGAAATCTGAGGCGGGAAGCAGAGACAGGCAAGCCTGTCACCTTGAAAGACGTCTTTGGGGCCTACAGCATGGATGTGATCACTAGCACATCATTTGGAGTGAACATCGACTCTCTCAACAATCCACAAGACCCCTTTGTGGAAAACACCAAGAAGCTTTTAAGATTTAATTCATTAGATCCATTCACTCTCTCAATAAAAATCTTTCCATTCCTTACCCCAATTCTTGAAGCATTAAATATCACTGTGTTTCCAAGAAAAGTTACAAGTTTTCTAACAAAATCTGTAAAACAGATAAAAGAAGTTCGCCTCAAAGAGACACAAAAGCACCGAGTGGATTTCCTTCAGCTGTTGATTGACTCTCAGAATTCGAAAGACACTGAGTCCCACAAAGCTCTGTCTGAACTGGAGCTCGTGGCCCAATCAATTATCTTTATTTTTGCTGGCTATGAAACCACGAGCAGTGTTCTCTCCTTCATTGTGTACGAACTGGCCACTCACCCTGATGTCCAGCAGAAACTGCAGAAGGAAATTGATACAGTTTTACCCAATAAGGCACCACCCACCTATGATACTGTGCTACAGATGGAGTATCTTGACATGGTGGTGAATGAAACGCTCAGATTATTCCCAGTTGCTATGAGACTTGAGAGGGTCTGCAAAAAAGATGTTGAGATCAATGGGATGTTCATTCCCAAAGGGGTGGTGGTGATGATTCCAAGCTATGTTCTTCATCATGACCCAAAGTACTGGACAGAGCCTGAGAAGTTCCTCCCTGAAAGGTTCAGTAAGAAGAACAAGGACAACATAGATCCTTACATATACACACCCTTTGGAACTGGACCCAGAAACTGCATTGGCATGAGGTTTGCTCTCGTGAACATGAAACTTGCTCTAATCAGAGTCCTTCAGAACTTCTCCTTCAAACCTTGTAAAGAAACACAGATCCCCCTGAAATTACACTTTGGAGGACTTCTTCGAACAGAAAAACCCATTGTTCTAATGGCTGAGTCAAGGGATGAGACTGTAAATGGAGCCTGA</CDS>TTTCCCTAAGGACTTCTGCTTTGCTCTTCAAGAAAGCTGTGTGCCAGAACACTAGAGACCTCAAATTACTTTGTGAATAGAACCCTGAAATGAAGACGGGCTTCATCCAATATACTGAATAAATAACTGGGGATTCTGTACAGGCATTGTGCTCTCTCATGGTCTGTGTAGAGTGTTATACTTGGTAATATAGAGGAGGTGAGCAAATCAGTGCTGGGGAAGTAGATTTGGCTCCTCTCCTTCTCATAGGACTATCTCCACCACCCCCAGTTAGCACCATTAACTCCTCCTGAGCTCTGATAATATAATAAACATTTCTCAACAATTTCAACCACGATCATTAATGAAAATAGGAATTATTTTGATGGCTGTAACAGTGACATTTATATCATATGTTACATCTGGAGTATTCTATAGTAAGTTTTATATTAAGCAAATCAATAAAACCTCTTTACAAAAGTATTATTGGATGCTTTCCTGCACATTAAGGAGAAATCTATAGAACTGAATGACTGAGAACCAACAACTAAGTATTTTGATCATTGTAATCACTGTTGGTGTGGGAACTGGAGTGCAGTGGTGCGATCTTGGCTCACTGCAAGCTCTGCCTCCCAGGTTCACGCCATTCTCCTGCCTCAGCCTCCTGAGTAGCTGGGATTACAGGCACCT</gene></seq><eos>"
seq9 = "RNA, Hippoglossus hippoglossus, {}<seq><gene>CGAGGCGCGCTCCCGGCAGTAGAAAGCAGCGCGCACGTCGCTGAGGCCCCACAGCTGCTCGGAGACACGAAGGATCAAAACATCCAGAAGTTACTGGTCCCGCAGCGGCTCCTCTCCAGAACCCAGTCCACCGCCAGTTCGGATCTAAACTCGACCCAGTGCTCATC<CDS>ATGGCGACCGCTGCGGTGTCCGCCCCCGCCGGATCCGGCCCCAGCCCCGGCTCGGCGGCCGAGCTCGTCCGCGGCCAGGCCTTCGACGTCGGGCCTCGGTACCACAACCTCTCGTACATCGGGGAGGGAGCCTACGGGATGGTGTGCGCCGCCCACGACCGGGAGAACAAGGTGCGCGTGGCCATCAAGAAGATCAGCCCCTTTGAGCACCAGACGTATTGTCAGCGCACCCTGAGAGAAATCAAGATCCTTCTGCGCTTCAAACACGAGAACATTATCGGAATCAACGACATCATCCGCACGCCGACCATCGACCTGATGAAGGACGTATACATCGTCCAGGATCTCATGGAGACCGACCTGTACAAGCTCCTGAAGACTCAGCACCTGAGCAACGACCACATCTGCTACTTCCTCTACCAGATCCTACGTGGGCTCAAGTACATCCACTCTGCCAACGTCCTGCACCGCGACCTGAAGCCTTCCAACCTCCTGCTCAACACCACCTGTGACCTCAAGATCTGTGATTTTGGTTTGGCTCGCGTGGCTGACCCCGACCACGATCACACTGGTTTCCTCACGGAGTACGTGGCCACGCGTTGGTACCGAGCACCTGAGATCATGCTCAACTCCAAGGGCTACACCAAGTCCATAGACATCTGGTCCGTGGGCTGCATCCTGGCGGAGATGTTGTCCAACAGGCCAATCTTTCCTGGGAAACACTACTTGGATCAGCTCAACCACATTTTGGGGATCCTCGGTTCTCCCTCTCAGGAGGATCTCAACTGCATCATCAACATCAAGGCCAGGAACTACCTGCTGTCGCTGCCCCTGCGCTGCAAAGTGCCGTGGAACCGCCTCTTCCCCAACGCCGACCCCAAAGCTCTGGACCTGTTGGACAAGATGTTGACCTTTAACCCTCACAAGAGGATCGAGGTGGAGGAGGCCCTGGCTCATCCTTACCTGGAGCAATACTACGACCCCACAGACGAGCCCGTCGCTGAGGCTCCGTTCAAGTTTGACATGGAGCTGGATGATCTACCCAAAGAGACACTGAAGGAGCTCATCTTTGAAGAGACTGCACGTTTCCAGTCCGCCTTCAGGTCCTAA</CDS>CGTCTCACACAGATCTCTGTGGACGGGCCTCTGCTCCTCCCTCTCCTC</gene></seq><eos>"
seq10 = "RNA, Neoarius graeffei, {}<seq><gene>CCCGAAGAGTACTTCACGACGCGCGCCCCCGCGTTTGCGTGTTCAGACTCGCGTGCACGCGTTATCCAATGTAGTACAGGAGGAAAAATCCGCGTTCACGTAAACATATACACTGATCGAAGCCAGGTGGACGTGGTATTGGTTTCTTTTCGACCGTCATTTCAAGTGCGACGTTCAAACAAGCCGTGATCGTTCACGGAAATCGGTAGAAGGCCCCCCCCTCCCTCCTTCCACGACCGAGCGGAACG<CDS>ATGGCGGAGTCGGGCGACACCGCAGAGGCGGTAACAGGAGCCGCGGGCTCGGAGGGCAGCGCCGCCGATACTGGCGGAACAGCCGCGCCTGATGGAGCCACTGGAGCCGCGGGACCCAAACCCGGCCTGGAGTCGGTGAAAGGACAAAATTTCGACGTGGGCCCGCGTTACACCGATCTCCTATACATCGGGGAGGGGGCCTATGGGATGGTCTGCTCTGCTTTTGACCATGTGAACAAGCTCAGAGTGGCCATTAAGAAGATCAGCCCATTTGAGCACCAGACATACTGCCAGCGCACGCTGCGAGAGATCAAGATCCTGCTGCGCTTCCGCCACGAGAACATCATCGGCATCAACGACGTCCTGAGAGCGCGCAACATTGACTACATGCGTGATGTCTACATCGTCCAGGACCTGATGGAGACGGACCTGTACAAGTTGCTGAAGACCCAACAGCTCAGCAACGATCACATCTGCTACTTTCTGTACCAGATCCTGCGTGGGCTGAAGTACATTCACTCTGCCAATGTCCTCCACAGAGACCTGAAGCCCTCCAACCTGCTCATCAACACCACCTGTGACCTCAAGATCTGTGACTTTGGGTTGGCACGGATAGCTGACCCAGACCATGACCACACTGGATTCCTCACAGAGTACGTGGCCACTCGCTGGTACCGTGCTCCCGAGATCATGCTCAATTCTAAGGGCTACACAAAGTCCATCGATATTTGGTCAGTGGGTTGCATCCTGGCTGAGATGTTGTCCAATCGGCCCATTTTCCCAGGGAAGCACTATCTAGACCAACTCAATCATATACTTGGCATCCTGGGCTCGCCTACTCATGAGGATCTAAGCTGTATAATCAACATGAAAGCCAGGAGCTACCTGCAGTCTCTCCCACATAAGCCCAAAATTGCCTGGAACAAGCTCTTTATCAAGGCGGATAGCAAAGCTCTGGATTTGTTGGACCGCATGTTGACCTTTAACCCCATGAAGCGTATAACGGTGGAAGAAGCACTGGCACACCCTTACCTGGAGCAGTACTATGACCCCACTGATGAGCCAGTGGCTGAGGAGCCCTTCACCTTTAACATGGAGCTAGATGATCTTCCTAAAGAGAAGCTGAAGGAACTCATCTTTGAAGAGACCGCACGATTCCAGGCAAACTACCAAGGTTCCTGA</CDS>GGGAAGCAAGCACCCAAGGAGAGCTCAAGCCACACTAAGGCTAAGAACAAGATTACTCCAAACAAGCTACAAAAGATGTACACACATGCACGCGCGCGCACACACGGTATCCAAAAAAAAAAAAAGAGGAAAAAAAACGTACATGGAAAACTTGAGCTTTTCTTTTCTTTCTCGCCTGGAGGCTCTGTTTGGGTTTCTAAACTGCTATCTTAACACCATCAAATGGGGGCAGCTGTATCCACCATGCCACTCCATGCTGCACACAGCCCCTACTCTTACTCTTCATGCTTATCTCCACACAATTTTTTTTTTTTCTTCCTTGCACTCTCTCATCTTGTCACTTTTGGCCTGTTAGTGGCCCTTTTTTTTTTTTTTAAATGATGGAGAAAGAATAGGGAGGGAGAGAATAAGCAAGCATGTCGTTACTGTGGGCTCCAGCTGGGTGTGTGTATATGTAATTTTGACTTTTTTTTTTCCTTCATTCTGAGTATTGTATGTCCAAGATCGATTTGTGCACTGTAATTTGAACGTTATGATCCTTTGCAAAATGTGTTTTGTATGTGGAGAGGGAGTTTGAGCATTTGTTAGGTTTTTTTTTTTTTTCCCTTTTTAGGTTGGTTTTCCTTGTGAGCTTTTTCAGCTCCGGAGTGTGAAAGCCAATGCTGGTGATTCCAAAAATATACAGGATGATGTATAAAACTGAATCTTGCTTTACTTACTGTACCATATCCAATTCCATTGATGGGTTTTCTTGAAACGCTCTCGCCAGTGCTCTTGGAAAATGAAGCCATCAGTATCCTTGATCTGAAATCTCATCCCCACACCCTTTTCTTTTTCTCGTTGTCTATCTCTCTGCTCTTAATTAGTTCATTTTTGCCACTGCACTCACTTTTTTTTTTTTTTAAACACCTTGACTGTTCATTTCCTCGATGTTGATTAGATTTTTCTCCCAGTTGTTGCTCTTACTTTTACAATAGCAGTCATCGATGGCTGACTTTAAGATGACTCTTCTCCCCTCCCCCTCTACCCATTAGTATATTTTATCGTAGATTTTAACCGCTGATCCCTTGACACCGCTTTCCTCCAGCATTACTTCACATTCTTTCTCCTCACGTTGCCAGTTGAGGAGGCCTAGCCAAGCCGAACGCTTTCAGAGCCATAGCACACACTCCTTCCCCCCCCCCGGTGATGTCACAAATTCTTCAAAAAGGGCTTTTTTTCCACGGGGCTTGCTGACCTCAATGCCTCTCTCGCCTGTGATTGGCTGACCTTGTGATAATGTAGTCTGTGACTGGCTGACTTTGCATGCACCGAATGGCAGGTAGTGTTTCTGAATTTCCTGCTCGTTTGAAGAAGTGTTATTGCCTCGTTTGTTTCCTGCAACTGACTAAACGCCTGTAACGATGTAGATATATGCTTTTCGTCAAGTCACTGATGAGGAAAGCTAGCTCTCGGTGGGGGAAATTGAGAAATTTTTAAATAGTGGGGGGGGGTTGTTTGTCTTTGAATAAATGCCGGGCCTGATTTCTACACTGATGTTCATGGTTCTGTTGATTTTATTACTGTGTGGGGAGGACAAAAGTGCACTCTATATAGCACGTTTCATACATCAGTCGAATGACGTTCTTCAATCTGTGCACCTGCACTTTGTCTCATTACCAAAAGCTTTTGCTTCTTCCCCCCTCCAGTGCAGTTCTGGTCTGTCCTCTTTGTAGAAACTTTTTTCTTCTCTGGCACTTGGCTTTTTCTCTTGCTTTGCTGGTGCATTGTGGGACAGGAAGGCACAGATGCTGTGTCTGCATGTCTGTCTGTTTCAGCTCTGAGCTGCTTTTTGCTCCGCGTTCTCTCGCTGCAAGTGACTCCACATCGCTCACTGAGCCGAGGAGAACAAATAATTACCAAGGAAAAATTTATGCCGCGTTGTTTGTTATTGATGTATCGCTTATCTACAAATGGGACCATAGGGACTGAACAATCACACAGCACACACCTACAGATATCCATCACTGACACACAGTATGAGTTGAGACCTACAAATGAGGCAGATTTAATAATGTCCTTTTAATGGCTGTTCTTATTATTGCTGTTATATTGAGTCTTTTTAAGAACTGAAAGGGATGTAAAACTTGTCTTTTTTTTTTTTTTTTTTTTCTTTTGCGTGTGTGTGTGAGAGAGAGAGAATTAATTGGGTGTGTCGTCAGTCTGTCGGAAATCTATTTTTCCCCTGAATCTTTTAAGGATTTATTTTAAAAGATGATTTAAAACTGAGTAGATTGTCAAGAAATATGAACCATGGATGGTTTGTGTATGTACATGCTTATGGGTATTATGAATTAAACAATTAACAAAGAAACAGACTTCTATACATATATTAAATAAAATTCCTGATACTGTAA</gene></seq><eos>"
seq11 = "RNA, Onychostoma macrolepis, {}<seq><gene>ATACTGTGGTGTACTGTACTGTAGTGTAGTATATTGTCAGTGTGAGTGTGCGGCGGGCTTTAGGACCCAGCGCGCCGCTCTCGCCGGACTCTCTCCGCAGGCTGGAGTCTCTCGCTGTGGGGAGTGTGTAGTGGAGGAGAGGCGAGGACACGCTCGTGCTCATCAAAACACTTTTACCTGTTTTCTGCGGGATTAAAGTAGCAACCGAAGCAGCGAGCTCCTCCACAGCAATGCGCCGGAGCCCTCAGTGAAACCCGTCTCGGCGACCCGTGTGAAGAGGCTTTACGCTGGCCGGATCGAAGCACTCGGTGGATCTGGATAGACTGACGGCCTGCCGCCGGAGAGC<CDS>ATGACGACGGACGTGGTGATCGTGAAGGAGGGATGGCTCCACAAGCGAGGAGAGTACATAAAGACGTGGAGGCCGCGGTATTTCTTGCTGAAGAGTGACGGTACGTTTATCGGCTATAAGGAGCGGCCGCAGGACGTGGATCAGCTGGAGACGCCGTTAAACAACTTCTCCGTGGCACAGTGTCAGCTGATGAAGACGGAGCGGCCCAAACCCAACACATTCATCATACGCTGCCTGCAGTGGACCACTGTCATTGAGCGCACCTTCCATGTGGAGAGCCCTGAGGAGCGGGAGGAATGGACCAAAGCCATCCAGACTGTAGCCGACAGTCTGCAGAAACAAGAGGAGGAGATGATGGACGCGTCTCCCGAACACATGGACATGGAGATGTTTCTGACCAAACCTCGGCTCAAAGTGACTATGCACGACTTCGAATACCTCAAACTGCTGGGAAAAGGCACTTTTGGGAAGGTGATTCTGGTGAAGGAAAAAGCAACGGGAAAATATTACGCCATGAAAATCCTCAAGAAAGAAGTGATTGTAGCCAAAGATGAAGTGGCTCACACACTCACAGAGAACCGTGTGCTGCAGAACTCCAAACACCCCTTCCTCACGGGTCTGAAGTACTCTTTCCAAACCCACGACCGCTTGTGTTTCGTCATGGAGTACGCAAACGGAGGAGAGCTCTTCTTCCACTTGTCGCGGGATCGGGTTTTCTCGGAGGACCGCGCGCGTTTCTACGGAGCAGAGATCGTGTCTGCGCTCGATTACCTGCATTCGGAGAGAAACGTGGTCTACAGGGACCTGAAGCTGGAAAATCTGATGCTGGATAAAGACGGTCACATAAAGATCACTGATTTCGGCTTGTGTAAGGAGGGAATCACAGATGGAGCCACGATGAAGACCTTCTGTGGGACTCCAGAGTATCTGGCTCCTGAGGTGCTGGAGGACAACGACTACGGCCGTGCAGTGGACTGGTGGGGTCTGGGAGTGGTGATGTACGAGATGATGTGCGGCCGTCTGCCGTTCTATAACCAGGATCACGAGCGTCTGTTCGAGCTCATCCTGATGGAGGACATCCGCTTCCCTCGAACGCTCACGCCCGAAGCCAAATCCCTGCTGTCCGGCCTGCTCAAGAAAGACCCCCAACAACGGTTAGGAGGCGGCCCGGATGATGCGAAAGAAATCATGCAGCACAAATTCTTCACTGGAATTGTGTGGCAAGATGTTTATGAGAAGAAGCTGGTTCCTCCGTTCAAACCGCAGGTGACCTCGGAGACGGACACCAGATATTTCGATGAAGAGTTCACAGGACAGACCATCACCATCACACCGCCGGGACAAGATGACAGTATGGAGTCGTTCGACAGCGAGAGGAGGCCGCACTTCCCACAGTTCTCCTACTCGGCCAGCGGAACGGCGTAA</CDS>AATATCAGCACAACTGCTCTGAGATCAGACACTCACCAGCACTGCACCGATTTACCACGTTTACCACGTCTGGATCTGTGATGTTCGTCCGTTCCACGTTGATTTCATGATGTTTTCTGAGGATATGAGAGGGAACACTAACATTTCCACTCAAACTGCCAAATTAATACAGAACTGATTGTGTAGCACTTCTTTAATCTGGGAACTAATGCCAAATATTACTGCATTTACATTCAAAAACGCCAATTTTATGAACTGACCAATTGAAACGTACAAAACGTACAGCAGAATAGCAGCTCTTATATGAATATCTAGAATGTAATTCTGCTACATTTGTAATTTTACACGAT</gene></seq><eos>"
seq12 = "RNA, Cyclopterus lumpus, {}<seq><gene>CCAAATGCGTACCTCGACTGTTGGAGAGGAGACAAGTTAACATAGCGGTGCTAAAGCTTGGCTAGCGTTTCTCTGTGCTGGCTCGTTGGTCGTACAATGAGAGGCGTTGTGCAAGAGCCGAATAGCAATTAAGTGGCCAACAGAATATAATAACAGAAACACATCAGGCCGTCCAA<CDS>ATGGTGATCATGTCGGGCACAGCCACTGTGCTGCAACAGTTTGTCAGCGGGCTGAAGAATCGAAATGAAGACACCAGAGCAAAATCTGCCAAAGAGCTGCAGCACTACGTGACCACAGAACTCCGAGAGTTAAGCCAAGATGAGGCCACTACATTCTACGATGAGTTAAACCACCACATATTCGAGCTGGTATCCAGCTCTGATGTGAACGAGAAGAAAGGAGGGATTCTCGCCATCGTGAGTCTGATCGGAGTCGAGGGAGGCAATGCCACCAGGATCAGCCGCTTTGCCAATTACCTGCGCAACCTGCTTCCGTCCAGCGACCCTGTGGTGATGGAGATGGCCTCCAAAGCCATGGGCCACCTGTCGATGGCAGGGGACACCTTTACTGCGGAGTATGTGGAGTTTGAGGTCAAGAGGGCCCTTGAGTGGCTGGGAGCAGACAGAAATGAGGGCAGACGCCATGCCTCAGTGTTGGTGTTGAGGGAGCTGGCTGTGAGCGCACCCACCTTCTTCTTTCAGCAAGTACAGCCCTTCTTCGACAACATCTTTTACGCAGTATGGGACCCCAAGCAGGCCATCCGAGAAGGGGCTGTATCTGCACTGCGAGCCTGCCTCATCCTCACCACGCAGAGGGAGACCAAGGAAATGCAAAAACCACAGTGGTACAAACAAACCTTCGAGGAGGCAGAGAAAGGCTTCGATGAGACTTTGGCCAAAGAGAAGGGGATGAATCGCGACGACAGGGTCCACGGAGCTCTCCTGATCCTGAATGAGCTGGTTCGCATCAGCAGCATGGAGGGAGAGCGCATGCGAGAGGAGATGGAGGAGATCACGCAGCAGCAGCTGGTCCATGACAAGTACTGCAAGGAGATGATGGGATTCGGAGCCAAGCCCCGACACATCACGCCCTTCACCAGCTTCCAGTCGGTGCAGCCGCAGCAGTCGAACGCCCTGCTGGGCCTGCTGGGCTACAGCACCCCTCAGGGCTTCCTCAGCTTCGGGGCCACTCCGGCACCCGCCAAGAGCTCGCTGGTCGAGAGTCGATACTGCCGCGAGCTGATGGAGGAACGATTTGACCAGGTGTGTCGGTGGGTACTGAAATACAAGACCAGTAAAAACCCTCTAATCCAGATGACCATCCTCAACCTGCTTCCTCGCCTGGCTGCCTTCAAGCCACATACTTTTACAGACCAGTACCTGTCGGACACCATGGGCTACCTGCTGGCCTGTCTGAAGAAGGAGAAGGAGAGGACAGCAGCTTTCCAGGCTCTCGGACTGCTGGTAGTGGCAGTCAGGGTAGAAATCCAGCAGTACCTCACGAAGATCCTTGAGATCATCAAGGCCGCCCTGCCGCCGAAGGATTTTGCCCACAAGAGGCAGAAGACCATGCAGGTGGACGCCACAGTGTTCACCTGCATCAGCATGCTTTCCAGAGCCATGGGTCCCAGCATCCAACCCGACATCAAGGAGCTGCTGGAGCCTATGCTGGCTGTGGGCCTCAGTCCGGCGCTGACGGCGGTGCTGTACGACCTGAGCCGTCAGATCCCCCAGCTGAAGAAGGACATCCAGGACGGCCTGCTGAAGATGCTCTCCCTGGTGTTGATGCACAAGCCTCTGCGTCACCCCGGCATGCCCAAAGGCCTGGCTCACCAGCTGGCCTCGCCAAGCCTCACAAACATCCCCGAGGCCAGCGACGTAGGCAGCATCACACTTGCCCTGCGTACACTGGGCAGCTTTGAGTTTGAAGGACACTCCCTCACCCAGTTTGTGCGCCACTGTGCCGATCACTTCCTGAACAGTGAACACAAGGAAATCCGGATGGAGGCAGCTCGGACCTGCTCGCGCCTGCTCACCCCCTCCATCCACATGATCAGCGGCCATGTTGTCAGCCAGACCGCTGTGCAGGTTGTCGCGGATGTTCTCAGCAAGCTGTTAGTTGTTGGCATCACCGACCCAGACCCAGACATCCGATACTGTGTGCTGGCATCGCTCGATGAGCGTTTCGACGCTCACCTTGCCCAGGCGGAGAACTTGCAGGCGCTGTTTGTGGCACTAAACGACGAGGTGTTCGAGATCAGAGAGCTGGCCATCTGCACAATCGGACGCCTCAGCAGTATGAACCCTGCCTTTGTCATGCCCTTCCTCCGCAAGATGCTCATCCAGATCCTAACGGAGTTGGAGCACAGTGGCGTCGGCAGGAACAAAGAGCAGAGCGCCCGTATGCTTGGTCATCTGGTCTCCAACGCCCCGCGCCTCATACGGCCGTATATGGAGCCCATCCTCAAGGCTCTCATCCTCAAGCTGAAAGACACAGACCCCAACCCTGGAGTGGTCATTAGTGTCCTCGCGACCATCGGAGAGTTGGCTCAGGTGAGTGGCCTGGAGATGAGGAAGTGGATGGACGAGCTTTTCCCAATCATCATGGACATGCTGCAGGACTCCTCCTCACTGGCCAAACGACAGGTGGCGCTGTGGACACTGGGCCAGCAGGTGGCTAGTACAGGTTATGTGGTTGAGCCCTACCGCAAGTACCCCTCCCTTCTAGATGTGCTGCTCAACTTCCTCAAGACGGAACAAAACCAGGGCATCAGGAGAGAGGCTATCCGTGTACTGGGTCTCCTCGGAGCTCTGGACCCCTACAAGCACAAGGTGAACATCGGCATGATCGACCAGTCCCGAGACGCCTCCGCCGTCAGTCTCTCTGAGTCCAAGTCCAGTCAGGACTCAGCGGACTACAGCACCAGTGAGATGCTGGTGAACATGGGCAACCTGCCCTTGGACGAGTTTTACCCGGCCGTGGCAATCGTCACTCTGATGCGCATCCTGAGGGACCCTTCACTCTCCAACCACCACACCATGGTGGTCCAGGCCGTCACCTTCATCTTCAAGTCTCTGGGCCTCAAGTGTGTCCAGTTTCTGCCGCAGGTCATGCCCACTTTCCTCAATGTGATCCGGGTCTGCGACGCCAGCATCCGAGAGTTCCTCTTCCAGCAGATGGGAATGGTGGTGTGCTTTGTGAAGGTCCACATCCGCCCGTACATGGACGACATCTTCACCCTCATCAGAGAGTACTGGACTCCAAACAACCCCATGCAGAACACCATCATCCTTCTGATCGAGCAGATTGTGGTGGCACTGGGTGGAGAGTTCAAGCTCTACCTGCCTCAGCTCATCCCCCACATGTTGCGTGTCTTCATGCACGACAACAGCACCGGGCGCAGTGTTACCATCAAGATGCTCAACGCCATCCAGCTGTTTGGTGCCAACCTGGACGACTACCTGCACTTGCTGCTGCCTCCCATCGTCAAGCTGTTTGATGCCTCCGACGTGACCCTGCAGGCCCGCAAGGTTGCCTTGGAGACACTGGACCGGCTGACTGAGTCCCTGGACTTCACCGACTACGCCTCACGCATCATCCACCCGATTGTTCGGACACTTGACAGCACGCCCGAGCTGCGCGCCACCTCCATGGACACCCTGTCCTCGCTCGTGTTCCAGTTGGGGAAGAAGTACCAGATCTTCATACCCATGGTGAACAAAGTGATGCTGAAGCACAGGATCAACCACCAGCGCTATGACATCCTCATATGTCGCATCGTTAAGGGCTACACTCTGGCTGAGGAGGAAGAGGACCCTCTAATCTTCCAACATCGCCAGCTTCGTGGTAACCAAGGCGATGCTCTGGTCAGCGGGCCGGTGGAGGCGGGGCCTATGAAGAAGCTTCACGTTAGCACCACTGCACTCCAGAAGGCTTGGGGTGCGGCCAGGAAGGTGTCCAAAGATGATTGGCTTGAGTGGCTGAGACGCTTGAGTGTGGTCCTGCTCAAGGAGTCGTCTTCTCCAGCCCTGCGATCATGCTGGTCTCTTGCCCAAACCTACATCCCTCTCGCCAGAGACCTGTTCAATGCCGCCTTCCTGTCTTGTTGGTCCGAGCTGAGTGAGGACCAGCAGGACGAGCTCATCCGCAGCATCGAGTTGGCCCTCACCTCCCAGGACATCGCTGAGGTCACCCAGACTCTGCTCAACCTGGCAGAGTTCATGGAGCACAGTGACAAGGGCCCTCTGCCCCTCAGAGACGATAACGGCATCGTGCTGCTCGGCGAGAGAGCCGCCAAGTGTCGCGCCTACGCCAAGGCTCTGCACTACAAAGAGCTGGAATTTCAGAAGGGGCCTTCTCCCCTCATCCTGGAGGCTCTTATCAGCATCAACAATAAACTACAGCAGCCAGAAGCCGCATCTGGGGTTCTGGAGTATGCCATGAAGCATTTTGGTGAACTGGAAATCCAAGCCACTTGGTATGAGAAGCTCCATGAGTGGGAGGACGCTCTGGTAGCGTATGACAAGAAGATTGACATGAACAAAGAGGATCCGGAGCTCATCCTGGGCAGAATGCGCTGCTTAGAGGCCCTGGGAGAATGGGGGCAGTTGCACCAGCAGTGCTGCGAGGAATGGGCGCTGGTGAGCGAGGAAACCCAGGCCAAGATGGCCCGTATGGCTGCTGCCGCCGCCTGGGGTCTAGGGCACTGGGACAGCATGGAGGAGTACACGTGTATGATCCCAAGAGACACACATGATGGCGCTTTTTATAGAGCGGTGCTCGCCCTGCATCAGGACCTCTTCTCATTGGCCCAGCAGTGTATTGACAAAGCAAGAGACCTCCTGGATGCCGAGCTGACGGCCATGGCCGGAGAGAGCTACAGTCGAGCCTACGGGGCCATGGTGTCTTGCCAGATGCTGTCAGAGCTGGAGGAGGTGATCCAGTATAAGCTGGTGCCAGAGAGGAGGGACATCATTAGGGAAACATGGTGGGAGAGGCTTCAGGGTTGTCAGCGCATCGTTGAGGACTGGCAAAGGATCCTGATGGTCAGATCGCTCGTCATCAGCCCCCACGAGGACATGAGGACTTGGCTGAAGTATGCCAGCCTCTGTGGCAGGAGCGGACGCCTGGCACTGGCCCATAAGACCCTGGTGCTCCTTCTGGGCGTCGACCCCTCCAAGCAACTTGATCACCCGCTGCCCACGGCCCACCCGCACGTCACCTACGCTTACATGAAGTACATGTGGAAGAGTGCCCGCAAGATTGATGCCTTCCAGCACATGCAGCACTTTGTACAAGGGATGCAGCAGCAAGCCCAGCACGCCATCGCTGCCGAGGACCAGCAGCACAAGCAGGAGCTCCACAAACTGATGGCCAGGTGTTTCTTGAAGCTGGGGGAGTGGCAGCTCAGCCTGCAGGGCATCAACGAGAGCACCATCCCCAAGGTTCTGCAGTATTACAGCCATTCCACTGAGCATGACCGCAACTGGTACAAGGCCTGGCACGCCTGGGCGGTGATGAACTTTGAGGCGGTGCTCCACTACAAACACCAGAACCAAGGACGGGACGAGCTGCGACACATTGGCGCCGCCAGTGCCAACAGCGAGGCCAGCAACAGCGACAGCGAGGTCGACAGCACCGACCACAGTCCTGTGCCCTCGCCGGGACAGAAGAAGGGCAACGAGCATATGCTGTGTGGTGAGCCTCCTCAGGACATCTCGAAGACGCTGCTCCTGTACACGGTTCCCGCCGTTCAAGGTTTCTTCCGCTCCATTTCCCTCTCCCGAGGCAATAACCTGCAGGACACACTCAGGGTTCTGACTCTGTGGTTCGACTACGGCCATTGGCCTGAAGTGAACGAAGCCCTGGTGGAGGGTATCAAGACCATTCAAATAGACACCTGGCTACAGGTGATCCCTCAGCTCATCGCTCGAATTGACACTCCTCGAGCCCTGGTGGGTCGCCTCATCCACCAGCTGCTCACGGACATCGGGCGGTATCATCCCCAAGCCTTGATCTACCCGCTAACGGTGGCCTCCAAGTCCACCACCACGGCCCGCCACAATGCTGCTAACAAGATCTTGAAGAACATGTGCGAGCACTGCAACGCTCTGGTTCAGCAGGCCATCATGGTGAGCGAGGAGCTCATCCGGGTGGCCATCTTGTGGCACGAGATGTGGCACGAGGGCCTCGAGGAGGCGTCGCGTCTTTACTTCGGCGAGCGCAACGTCAAGGGCATGTTTGCTGTGCTGGAGCCGCTCCATGCCATGATGGAGAGGGGACCGCAGACCCTCAAGGAGACCTCTTTTAACCAGGCTTACGGCAGGGACTTGATGGAGGCTCAGGACTGGTGCAGGAAGTACATGCGCTCTGGAAACGTGAAGGACCTGACCCAGGCCTGGGACCTCTACTACCATGTGTTCAGACGCATTTCCAAACAGCTGCAACAGCTGACTTCCCTGGAGCTACAATATGTGTCTCCAAAGCTGCTGATGTGCCGGGATCTGGAGCTGGCAGTACCAGGCACCTATGACCCCAACCAGCCTATCATCCGCATCCAGTCCATCGCCGCCTCCCTGCAGGTCATCACCTCAAAGCAGCGCCCACGCAAACTCACCATCATGGGCAGCAACGGCCACGAGTTCATGTTCCTGTTGAAGGGCCATGAGGACCTGAGGCAGGATGAGAGGGTCATGCAGCTGTTTGGTCTGGTCAACACACTGCTGGCCAACGACCCGGCGTCCTTGCGCAAGAACCTCAGCATTCAGCGCTACGCAGTGATCCCCCTGTCCACCAACTCAGGCCTGATTGGCTGGGTGCCCCACTGTGACACCCTGCATGCCCTCATCAGAGACTACAGAGAGAAGAAAAAGATTCTGCTCAACATTGAACATCGCATCATGCTCAGGATGGCCCCAGACTACGACCACCTGACGCTGATGGAGAAGGTGGAGGTGTTCGAGCACGCCGTCAACAACACGGCCGGAGACGACCTGGCCAAGCTCTTGTGGCTGAAGAGCCCCAGCTCTGAGGTGTGGTTCGACCGGAGGACCAATTACACCCGCTCCCTGGCTGTGATGTCCATGGTGGGCTACATCCTCGGACTGGGTGACAGGCATCCCTCCAACTTGATGTTGGACCGGTTAAGTGGAAAGATCCTCCACATTGACTTTGGAGACTGTTTTGAGGTCGCCATGACCAGAGAGAAGTTCCCTGAAAAGATCCCATTCCGACTAACGAGGATGCTGACCAACGCCATGGAGGTGACGGGCTTGGACGGTAACTACCGGATCACCTGCCACACGGTGATGGAGGTGCTGAGGGAGCACCGGGACAGCGTCATGGCCGTGCTGGAGGCCTTCGTCTACGACCCGCTGCTCAACTGGCGGCTCATGGACACGAACACCAAAGGCAACAAGCGTTCCCGCACCAGGACCGACTCGTACACCGCCGGGCAGTCAGTGGAGGCCCTTGAGGGAATCGACCTGGGGGAGACCACACACAAGAAACCCGGCAGCACGGTGCCCGAGTCCATCCACTCCTTCCTCGGAGACGGCACGGTTCAACCTGAGGCTCTCAACAAGAAGGCCATCCAGATTATCAACAGAGTGAGAGACAAACTTACGGGTCGAGACTTCTCTCGCGATGAGACGCTGGACGTGCCAACACAAGTGGAGCTCCTCATCAAGCAAGCCACGTCACATGAAAACCTGTGCCAGTGCTACATTGGATGGTGTCCATTTTGGTGA</CDS>AGAACCGCCTGCGGCAGACAAAGATAAACTTGACTTTTCTTTATTTGTTCTCACTGGTAAGCCAAGAAGAAGCACTCAACTCGGATATTCATCCCCAACAATGTGCTTCAGCGACACAGTGCCGAGCCTTTTGAGAAAGACATATTGTTTTGTTAATGTTGTAGAATTATCTGATTTTTCCAGTGGCATGTTTTTAAGGTTATGAGAAAAAAAAAGCTCACAGTTTTGGACCTATTGATCACAATTGAACTCAAATGACTTTCTTCTCTTTTAGCACCAGAATGAATTTGCTCAAGCTTTCTACAGTGGCCATATTGTAATAAGTTATGACAATTCTTACTTCACACGCTGTATCAAAAACCCACGAGTCCGTTGTATCATCGTGTATTAATGTGACCGATTTCAATAACTAACTTAAACATTCTTATTAAGTTTACTCGTGACAAGTCACATTTGATTTGAGATGATCATAATGTACCGTACGTGTGCCATTAAACCCTGTACAGAAACC</gene></seq><eos>"
seq13 = "RNA, Oncorhynchus mykiss, {}<seq><gene>CGCAGTTCGTGCGGGATTATATCATTTACCCTTGAAACACCCCCCTCCCAAAACCAAAAGTTCAAA<CDS>ATGGAAGATGAAATCGCCGCACTGGTTGTTGACAACGGATCCGGTATGTGCAAAGCCGGATTCGCCGGAGATGACGCGCCTCGGGCTGTCTTCCCCTCCATCGTCGGGCGTCCCAGGCATCAGGGAGTGATGGTTGGGATGGGCCAGAAAGACAGCTACGTGGGAGACGAGGCTCAGAGCAAGAGGGGTATCCTGACTCTGAAGTACCCCATCGAGCACGGCATCGTCACCAACTGGGACGACATGGAGAAGATCTGGCATCACACCTTCTACAACGAGCTGCGAGTGGCTCCAGAGGAGCACCCCGTCCTGCTCACAGAGGCCCCCCTCAACCCCAAAGCCAACAGGGAGAAGATGACCCAGATTATGTTTGAGACCTTCAACACCCCTGCCATGTACGTGGCCATCCAGGCCGTGTTGTCCCTGTACGCCTCTGGCCGTACCACCGGTATCGTCATGGACTCCGGTGACGGCGTGACCCACACAGTACCCATCTACGAGGGCTACGCTCTGCCCCACGCCATCCTGCGTCTGGATCTGGCCGGCCGCGACCTCACAGACTACCTGATGAAGATCCTGACGGAGCGCGGTTACAGCTTCACCACCACGGCCGAGAGGGAAATCGTACGAGACATCAAGGAGAAGCTGTGCTACGTGGCGCTGGACTTTGAGCAGGAGATGGGCACCGCGGCCTCCTCTTCCTCTCTGGAGAAGAGCTACGAGCTGCCTGACGGACAGGTCATCACCATCGGCAACGAGAGGTTCCGCTGCCCAGAGGCCCTCTTCCAGCCCTCCTTCCTCGGTATGGAGTCTTGCGGTATCCACGAGACCACCTACAACTCCATCATGAAGTGTGACGTGGACATCCGTAAGGACCTGTACGCCAACACGGTGCTGTCCGGAGGAACCACCATGTACCCCGGCATCGCTGACCGGATGCAGAAGGAGATCACCTCTCTGGCCCCCTCCACCATGAAGATCAAGATCATCGCCCCCCCAGAGCGTAAATACTCCGTCTGGATCGGAGGCTCCATCTTGGCTTCTCTCTCCACCTTCCAACAGATGTGGATCAGCAAGCAGGAGTACGACGAGTCTGGTCCCTCCATCGTCCACCGTAAATGCTTCTAA</CDS>ACAGACTGTACCCATCCCAAACGACCGACCCAGCCACACCCGACTACCACTTCAGCTCTGCCACCAGACACACCACCCAGAGGGAGAGAGAGAGGGGGCCCGCGGAGTGAGCCCAAACCCAGCTTCTCAGTCTCATTGGCATGGCTTCTGTTATTTTGGCGCTTTTTTTTGACTCAGGATCTGTGAAAAACAAAAAAATATATACACAACTGGAACGGTGAAACAGCGGATTAAAGATTTTTAAGAGTTAAAAAAATAATGTAAGGACCCCTGAGATTCTATCATGCAGCTGAGGACTTTAAAAACAACAACATGTACATTCTGTTTCTTTTTCGAGTCATTCCAATTGTTTTGTTAACTTTTCAGAACAGGTATTTGCCTCTGTGAAGGCTGCCGGCTTCCGGCTCTCTGCCACAGCGTTCTGTAACATGGGGCAGTATGGCTTGTATGTAAATTATGTCACAATGTCTGGGTTTTTTTGTACTTGCAGCCTTAAGTCTTGGTCCTGTTTAATTTTTTTTTGTTTTTGTTTATGCAAATGAACCCGAGTGTGACCTCTGCTGGGGGGGGTCAGAGGTGATTAGGGTGCCAGAGGGACCAGTGGGGGGGGGGGCCAACTTGTACACTGACTAAACTCCCAATAAAAGTGCACATGTGTTCCGACATCAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA</gene></seq><eos>"
seq14 = "RNA, Xenopus tropicalis, {}<seq><gene>CAGAACGTCGCTGTCTCCAACGTTCCATCCTCCGAACATCTATTACGCCTGAATC<CDS>ATGAGGGAGATTGTACACATCCAGGCCGGCCAGTGTGGCAACCAGATCGGCGCTAAGTTTTGGGAGGTGATCAGTGACGAACATGGTATTGACCCCACTGGCACCTATCATGGAGACAGTGATCTGCAGCTTGACAGAATCAGCGTCTATTACAATGAGGCCACAGGTGGTAAATATGTCCCTCGAGCCATCCTTGTTGATCTGGAACCAGGAACTATGGACTCTGTGCGCTCAGGACCCTTTGGACAGATATTCAGGCCTGACAACTTTGTCTTTGGTCAAAGTGGTGCTGGCAACAACTGGGCAAAAGGCCATTACACAGAAGGTGCTGAATTGGTAGACTCTGTACTGGATGTTGTGAGAAAAGAGGCAGAGAGCTGTGACTGCCTACAGGGCTTCCAGCTCACTCACTCTCTGGGTGGTGGAACAGGGTCTGGTATGGGCACTCTCTTGATTAGTAAGATCCGAGAAGAATACCCTGACCGCATCATGAATACCTTTAGTGTTGTCCCATCTCCAAAGGTGTCTGATACAGTGGTAGAACCATACAATGCAACTCTTTCTGTTCATCAGCTAGTTGAAAACACAGATGAAACCTACTGTATTGATAATGAAGCACTCTATGACATCTGCTTCCGTACCCTCAAGCTTACTACACCAACATATGGAGACCTTAATCATCTTGTTAGTGCTACAATGAGTGGAGTCACTACCTGCCTCCGTTTCCCTGGCCAGTTAAATGCTGATCTTCGCAAATTGGCTGTCAACATGGTGCCCTTCCCACGTCTACACTTTTTCATGCCTGGGTTTGCGCCCCTAACCAGCAGAGGCAGTCAGCAGTACCGTGCACTGACAGTGCCTGAGCTCACACAGCAGGTGTTTGATGCCAAAAACATGATGGCAGCCTGTGACCCACGCCACGGCAGGTACCTTACAGTGGCAGCTGTATTCCGTGGACGTATGTCCATGAAAGAAGTTGATGAGCAGATGCTTAATGTCCAGAACAAGAACAGCAGCTACTTCGTTGAATGGATTCCCAACAATGTCAAGACAGCTGTATGCGACATACCACCCCGTGGCCTTAAGATGGCAGTCACCTTCATTGGTAACAGCACTGCCATTCAGGAGCTGTTCAAACGCATCTCAGAGCAGTTTACAGCCATGTTCCGTCGCAAGGCCTTCTTGCATTGGTACACAGGAGAAGGGATGGATGAGATGGAGTTCACAGAAGCGGAGAGCAACATGAATGATCTGGTCTCTGAGTACCAGCAGTATCAAGATGCTACTGCTGAAGAAGAGGAAGATTTCAATGAAGAGGCTGAAGAAGAAGCTTAG</CDS>ACTCTTCTCACTGTTGCTGAAATTCCTTTCACGTTTCCTCCTCCATTCTCAGGCTGAGAAAGATCTGTACCGAAGAGTAGTATTTGTTTTATATTCATATTTTTCCATTTCCTTTTTTCTCCTTACTTGTTTTTGTTGTCTCTCAAACTTCTATTCCCATTTTCTGTGTTCTGCTTGTCAGTTAAGGTAGGTTGTGGCTTAATTGGGGTAAACAGTTTGCTAAGGTTTTACAGTAGGTCTCTGGTTTCAGTATGCAGTAATTTGCCTCTGGATTTGGGTTTTGGATTTTAAGAGAGTTTTAACAGCCCATTGTGAATTCCATCTTTCCTTGGCCTAATAACATCATCAAATTTGTGGCTGGGTGCTGGTTTCTGCAGCTACATTTGGGTTACAAAGCAAACCTCTAATTTAGTCCCTTGGGACTTTACAAAACCCTTGCAGATTTTGCTTCTTTCTGTGCTTTTACAGATCACATTCAAGTGTTTGGTGGACATTCACATTTTCTTATGTTACTATGGAAATGACCTAGTAAGTTTTAAAATACACGTAACTGCAACTTCTGTCAACTAGAGCAATAGATATGATTCGGTTTTGCTCACATTTATGTAAATCCCAAGTATTTGAAATACCTTAATACTTCCAATTAGAAAATATGAAATGAAACTGATGTTTAACAAATGCACTTAATTCCCTTTACTGTAAACAGTATTGGTTGGTTAATTATAAGCTACATTAATCATATAGGCCCACTATCCATTGGGTAACTGGCATGGTTCATATTGCCTGCAGTTGTATGTAGAGAAGTGCTCTGCATTTTACTTAACAAGTAACAGAAAAAAGTGCAGAAGAATGCTAAAGAACTCTACACAGTGATATAGGGATTATTTCTTCTTCTCTTAACTTTACTCTTCAGATTTTCTTACATTACTTCCCTTTTTCTTCTCTTGTTCCCATACCATTATTTTTATTTTACTGTTGCTTTTTCATATTGAAAAGATGACAAGCCTGTGACGAAAAAAAATAAAACCCTTTTTTTTTTAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA</gene></seq><eos>"
seq15 = "RNA, Pezoporus wallicus, {}<seq><gene>CTTCGCCCGTCGTGCCGCCCAGCAGCTCTGCTCCGGTCCCGGATTACAAAGCCGAACCGCCGCCGCC<CDS>ATGAGCATCAGCACTAAGAACTCGTCGTACCGCCGCATGTTCGGCGGGGGCACCCGCCCCAGCACCGGCACCCGCTACATCACCTCCAGCACCCGCTACTCGCTGGGCAGCTCGCTGCGACCCAGCACCGCCCGGTACGTGTCCGCCTCGCCCGGAGGCATGTACGCCAGCAAGACGACGTCGGTGCGGCTGCGGAGCAGCATGCCGCCCATGCGGCTCCACGACTCCGTGGACTTCTCCCTGGCCGACGCCATCAACACGGAGTTCAAGGCGAACCGCACCAACGAGAAGGTGGAGCTGCAGGAGCTCAATGACCGCTTCGCCAACTACATCGACAAGGTGCGCTTCCTCGAGCAGCAGAACAAGATCCTCCTGGCCGAGCTGGAGCAGCTCAAGGGCAAGGGCACGTCCCGCCTGGGCGACCTGTATGAGGAGGAGATGCGGGAGCTGCGGCGGCAGGTGGACCAGCTCACCAACGACAAGGCACGCGTGGAGGTGGAGCGGGACAACCTCGCCGACGACATCATGAGGCTGAGGGAGAAGTTGCAAGAGGAGATGCTTCAGAGGGAGGAAGCTGAGAACACCCTGCAGTCCTTCAGACAGGATGTTGACAATGCCTCTCTGGCACGTCTTGACCTTGAGCGCAAAGTTGAGTCCCTGCAAGAGGAGATTGTCTTCTTGAAGAAGCTTCATGATGAGGAAATCCGAGAACTGCAGGCCCAACTCCAGGAACAGCACATCCAAATTGATATGGATGTTTCTAAGCCTGATCTTACTGCTGCCCTGCGTGACGTTCGCCAACAGTATGAAAGCGTTGCTGCTAAGAATCTTCAGGAAGCTGAAGAGTGGTACAAGTCCAAATTTGCAGATCTCTCCGAAGCTGCTAATAGGAACAACGATGCCCTGCGCCAGGCCAAACAAGAAGCTAATGAGTACCGCAGACAGATTCAGTCTCTCACCTGCGAAGTTGATGCCCTTAAGGGAAGCAATGAATCCCTGGAGCGCCAAATGCATGAAATGGAGGAGAATTTTGCTGTCGAAGCTGCTAACTACCAAGACACTATTGGCCGTCTGCAGGATGAGATTCAGAACATGAAGGAAGAAATGGCTCGCCATCTCCGCGAGTACCAGGACCTGCTGAATGTAAAGATGGCTCTTGATATCGAGATTGCCACCTACAGAAAACTGCTGGAGGGTGAAGAGAGCAGGATTAACATGCCTATTCCAACCTTTGCTTCTTTGAACCTGAGAGAAACTAACATTGAGTCTCAGCCAATGGTTGACACTCACTCAAAGAGGACACTTCTAATTAAGACTGTCGAAACTAGAGATGGACAGGTTATTAATGAGACTTCCCAGCATCATGATGACTTGGAGTAA</CDS>AGCTGAAGTGAAGATGCATACTTATAAGTGCAGGAGAAATTCTTACCAGCAAGATTTAAAAAGTCCATGTCTTAAAGGAAGAAACAGCTTTCAAGTGCCTTTCTGCAGTTTTTCCATGAGCGCAAGATTATTATGCTAGGAAATAGGTCTTAGATCTTGCAAACTGACTCTCCCTGAAAGATTAGAGTTTACAATGGAGTCTAGTTTACAAATAGCAATATCTTGTGCTGCAATACTGTTTTTAAGTATCTGAATTCAATAAAACTGCTTTTTCCAGCACAGTATGAGCAACCTGTCGCTACTTCAATAAATCTTTGGAAAATGGCTATTGATGTGTTCTAATTTGTTAACTTTATGACTTTCTGGAAAGCCATAACATCATAATGCTGGAATTACTGTACGGTTGACATCTAGTACTGGTTGTGTGGATTGCTGTTTTTTTCAGTTTAACTAGATAAACTGTCTTACTCATTTACTGCTTAGGTTTTGGAACCAACTAAAATGGACTATAACTGGCAGATGCATAATGTATTATGATATTTCTTATCACTTGAATAAAATCTGATACTTCAAGCTAATAAAAATTAATCTTGCTTTCA</gene></seq><eos>"

seqs = [seq1, seq2, seq3, seq4, seq5, seq6, seq7, seq8, seq9, seq10, seq11, seq12, seq13, seq14, seq15]

annotations = [
    "HBA1, hemoglobin subunit alpha",
    "HBB1, hemoglobin subunit beta-1",
    "MB, myoglobin",
    "CYGB2, cytoglobin-2",
    "CYP1A2, cytochrome P450 1A2",
    "CYP2C8, cytochrome P450 2C20",
    "CYP2D6, cytochrome P450 2D6",
    "CYP3A7, cytochrome P450 3A7 isoform X4",
    "MAPK1, mitogen-activated protein kinase 1",
    "MAPK3, mitogen-activated protein kinase 3",
    "AKT1, RAC-alpha serine/threonine-protein kinase",
    "MTOR, serine/threonine-protein kinase mTOR",
    "ACTB, actin beta",
    "TUBB, tubulin beta chain",
    "VIM, vimentin"
]

display_names = [anno.split(",")[0].strip() for anno in annotations]


In [ ]:
results = []
for i, seq_template in enumerate(seqs):
    orig_display = display_names[i]
    
    for j, replace_anno in enumerate(annotations):
        replace_display = display_names[j]
        
        seq_filled = seq_template.format(replace_anno)
        
        seq_tokens = tokenizer.encode(seq_filled)
        res = sequence_likelihood_no_bos_short(model, seq_tokens, device=device)

        results.append({
            "origin": orig_display,
            "replace": replace_display,
            "total_logprob": res["total_logprob"],
            "perplexity": res["perplexity"],
        })
df = pd.DataFrame(results)


In [ ]:
mat_perplex = df.pivot(index="origin", columns="replace", values="perplexity")

mat_perplex = mat_perplex.reindex(index=display_names, columns=display_names)

baseline_ppls = np.diag(mat_perplex)

mat_delta = mat_perplex.sub(baseline_ppls, axis=0)

mat_delta.head()

mat_delta_norm = mat_delta.div(mat_delta.max(axis=1), axis=0)


In [ ]:
plt.figure(figsize=(8, 7), dpi=300)

ax = sns.heatmap(mat_delta_norm,           
                 cmap="RdYlBu",            # 【关键修改】使用 RdYlBu (0=红, 0.5=黄, 1=蓝)
                 vmin=0, vmax=1,           
                 linewidths=0.5,      
                 linecolor='white',
                 square=True,
                 annot=True,               
                 fmt=".2f",                
                 annot_kws={'size': 10},    
                 cbar_kws={'shrink': 0.8}
                 )

ax.set_xticklabels(mat_delta_norm.columns, rotation=90, fontsize=10, fontstyle='italic')
ax.set_yticklabels(mat_delta_norm.index, rotation=0, fontsize=10, fontstyle='italic')

ax.set_xlabel("Provided Gene Symbol (Prompt)", fontsize=14, labelpad=10)
ax.set_ylabel("Real Gene Symbol", fontsize=14, labelpad=10)

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=10)
cbar.set_label('Normalized $\Delta$ Perplexity', fontsize=14) 
cbar.outline.set_visible(False)

plt.tight_layout()
plt.show()
